# DaViT-T + Gated Fusion

**Fusion family:** Gated  
**Mechanism:** per-modality sigmoid gates, directly inspectable as an XAI signal  
**Designed in this work:** yes  
**New on ISIC-DICM-17K:** yes

Every hyper-parameter comes from the shared `config.py`. The only per-model setting is the micro-batch size, which is compensated by gradient accumulation so the effective batch is 32 for all models.

**Before running:** attach the splitted-data dataset, set the accelerator to GPU, and turn Internet on so pretrained weights can download.

## 1. Environment

In [ ]:
# Dependencies. Kaggle images already ship torch/timm/transformers; these pins
# only add what may be missing. Keep quiet to avoid flooding the log.
!pip install -q "timm>=1.0.9" "transformers>=4.40" shap lime optuna 2>/dev/null
import torch, timm, sys
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("timm", timm.__version__)


## 2. Unified framework

In [ ]:
# The unified framework, written out verbatim so this notebook is
# self-contained and reproduces the local runs exactly.
import os, pathlib
os.makedirs("/kaggle/working/unified", exist_ok=True)
FILES = {}

FILES['config.py'] = r'''"""
Unified experimental configuration -- SINGLE SOURCE OF TRUTH.

Every architecture in this study is trained and evaluated through this file.
Nothing below may be overridden inside a model notebook; the only per-model
degree of freedom is `micro_batch`, a pure memory/throughput knob that is
compensated by gradient accumulation so the EFFECTIVE batch size is identical
for every model.

Rationale for each shared value is documented inline so that the Methodology
section of the paper can cite this file directly.
"""
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any
import json

# ---------------------------------------------------------------------------
#  Reproducibility
# ---------------------------------------------------------------------------
SEED = 42

# ---------------------------------------------------------------------------
#  Data split -- FIXED, PRE-COMPUTED, SHARED BY EVERY MODEL
#  Produced once by "EDA & preprocessing/final-splitted-data.ipynb" using a
#  stratified 70/15/15 partition with random_state=SEED. Models consume these
#  exact files; no model is permitted to re-split the corpus.
# ---------------------------------------------------------------------------
SPLIT_RATIOS = (0.70, 0.15, 0.15)
N_TRAIN, N_VAL, N_TEST, N_TOTAL = 11_941, 2_559, 2_559, 17_059

TARGET_COL = "class"
IMAGE_COL = "image_fixed"          # filename inside the images/ folder
ID_COLS = ["image", "isic_id", "patient_id", "year", "image_fixed"]

# ---------------------------------------------------------------------------
#  Metadata feature sets (ablation axis)
#
#  LEAKAGE WARNING, measured on the training split:
#    diagnosis_confirm_type_serial imaging showing no change -> P(mel)=0.000 n=1687
#    diagnosis_confirm_type_single image expert consensus    -> P(mel)=0.000 n= 851
#    diagnosis_confirm_type_Unknown                          -> P(mel)=0.000 n= 351
#    diagnosis_confirm_type_confocal microscopy w/ consensus -> P(mel)=0.000 n=  63
#
#  The confirmation-method field deterministically encodes the label for ~25%
#  of the corpus, because melanoma ground truth was only ever assigned to
#  biopsy-confirmed lesions. A model given this field learns the acquisition
#  protocol, not dermatology. FULL is reported for comparability with prior
#  work; NO_CONFIRM is the headline, leakage-controlled configuration.
# ---------------------------------------------------------------------------
CONFIRM_PREFIX = "diagnosis_confirm_type"
AGE_PREFIXES = ("age_scaled", "age_group")

FEATURE_SETS: Dict[str, Dict[str, Any]] = {
    "full":       {"drop_prefixes": ()},
    "no_confirm": {"drop_prefixes": (CONFIRM_PREFIX,)},
    "no_age":     {"drop_prefixes": AGE_PREFIXES},
    "no_both":    {"drop_prefixes": (CONFIRM_PREFIX,) + AGE_PREFIXES},
}
DEFAULT_FEATURE_SET = "full"       # main table; ablations are separate runs


# ---------------------------------------------------------------------------
#  Shared training protocol
# ---------------------------------------------------------------------------
@dataclass
class UnifiedConfig:
    # -- identity ----------------------------------------------------------
    model_name: str = "vit_early_fusion"
    feature_set: str = DEFAULT_FEATURE_SET
    run_tag: str = ""

    # -- image pipeline (IDENTICAL for every architecture) -----------------
    img_size: int = 224        # every backbone is instantiated at 224x224
    resize_to: int = 256       # shortest side, then centre-crop to img_size
    mean: List[float] = field(default_factory=lambda: [0.485, 0.456, 0.406])
    std: List[float] = field(default_factory=lambda: [0.229, 0.224, 0.225])

    # -- optimisation (IDENTICAL) ------------------------------------------
    effective_batch: int = 32  # held constant via gradient accumulation
    micro_batch: int = 32      # ONLY per-model knob (VRAM); see MODEL_ZOO
    epochs: int = 20
    warmup_epochs: int = 2
    lr_head: float = 3e-4
    lr_backbone: float = 3e-5  # discriminative LR: 10x lower on pretrained weights
    min_lr: float = 1e-6
    weight_decay: float = 0.05
    grad_clip: float = 1.0
    label_smooth: float = 0.05
    ema_decay: float = 0.999
    early_stop_patience: int = 6
    monitor: str = "val_auc"

    # -- runtime -----------------------------------------------------------
    amp: bool = True
    channels_last: bool = True
    num_workers: int = 4
    seed: int = SEED
    deterministic: bool = False   # cudnn.benchmark instead, for throughput

    # -- evaluation protocol (IDENTICAL) -----------------------------------
    threshold_rule: str = "youden"  # chosen on VALIDATION, applied to TEST
    calibrate: bool = True          # temperature scaling fitted on VALIDATION
    tta: bool = False               # off for the main table (fair comparison)
    n_bootstrap: int = 1000         # 95% CI on test metrics

    # -- XAI (IDENTICAL) ---------------------------------------------------
    xai_sample_strategy: str = "single"  # ONE representative case per figure
    xai_n_samples: int = 1
    ig_steps: int = 50
    lime_n_samples: int = 1000
    # KernelExplainer evaluates (shap_nsamples x shap_background) forward passes
    # with the image held fixed. These values keep that product near 4k, which
    # is a few minutes even for SegFormer-B5; raising them improves the
    # attribution estimate at roughly linear cost.
    shap_background: int = 25
    shap_nsamples: int = 150

    # -- paths -------------------------------------------------------------
    data_root: str = "/kaggle/input"
    out_root: str = "/kaggle/working/outputs"

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2)


# ---------------------------------------------------------------------------
#  Model zoo -- the 14 architectures of the benchmark.
#
#  micro_batch       tuned so the model fits a 16 GB T4 under AMP; gradient
#                    accumulation restores the common effective batch of 32.
#  origin            "ours"    -> fusion mechanism designed by this work
#                    "adapted" -> standard fusion with a standard backbone
#  novel_on_dataset  backbone not previously applied to ISIC-DICM-17K
# ---------------------------------------------------------------------------
MODEL_ZOO: Dict[str, Dict[str, Any]] = {
    "vit_early_fusion": dict(
        display="ViT-B/16 Early Fusion", family="Early fusion",
        backbone="vit_base_patch16_224", micro_batch=32,
        origin="ours", novel_on_dataset=False,
        note="metadata projected into token space, prepended before block 1"),

    "hybrid_vit_mlp": dict(
        display="Hybrid ViT-MLP (Late Fusion)", family="Late fusion",
        backbone="vit_base_patch16_224", micro_batch=32,
        origin="adapted", novel_on_dataset=False,
        note="independent encoders, concatenation at the classification head"),

    "convnext_bicross": dict(
        display="ConvNeXt-T + Bidirectional Cross-Attention", family="Cross-attention",
        backbone="convnext_tiny", micro_batch=32,
        origin="ours", novel_on_dataset=False,
        note="mutual image<->metadata attention, 2 stacked layers"),

    "swinv2_tabtransformer": dict(
        display="Swin-V2 + TabTransformer Cross-Attention", family="Cross-attention",
        backbone="swinv2_cr_small_224", micro_batch=16,
        origin="ours", novel_on_dataset=False,
        note="shifted-window features queried by clinical tokens"),

    "gnn_maxvit": dict(
        display="MaxViT-T + Graph Fusion (GATv2)", family="Graph",
        backbone="maxvit_tiny_tf_224", micro_batch=16,
        origin="ours", novel_on_dataset=True,
        note="heterogeneous image/metadata node graph, GATv2 message passing"),

    "segformer_laca": dict(
        display="SegFormer-B5 + Lesion-Aware Cross-Attention", family="Cross-attention",
        backbone="nvidia/mit-b5", micro_batch=8,
        origin="ours", novel_on_dataset=True,
        note="LACA: clinical CLS token gates per-patch saliency before fusion"),

    "davit_gated": dict(
        display="DaViT-T + Gated Fusion", family="Gated",
        backbone="davit_tiny", micro_batch=32,
        origin="ours", novel_on_dataset=True,
        note="per-modality sigmoid gates, directly inspectable as an XAI signal"),

    "maxvit_crossattn": dict(
        display="MaxViT-T + TabTransformer Cross-Attention", family="Cross-attention",
        backbone="maxvit_tiny_tf_224", micro_batch=16,
        origin="adapted", novel_on_dataset=True,
        note="multi-axis attention backbone, standard cross-attention fusion"),

    "effnetv2_crossattn": dict(
        display="EfficientNetV2-S + TabTransformer Cross-Attention", family="Cross-attention",
        backbone="tf_efficientnetv2_s", micro_batch=32,
        origin="adapted", novel_on_dataset=False,
        note="convolutional control for the cross-attention family"),

    "dinov2_crossattn": dict(
        display="DINOv2 ViT-S/14 + Bidirectional Cross-Attention", family="Cross-attention",
        backbone="vit_small_patch14_dinov2.lvd142m", micro_batch=16,
        origin="ours", novel_on_dataset=True,
        note="self-supervised features, no dermoscopic supervision in pretraining"),

    "biovil_lcsa": dict(
        display="BioViL-T + Lesion-Conditioned Sparse Attention", family="Sparse cross-attention",
        backbone="vit_base_patch16_224", micro_batch=16,
        origin="ours", novel_on_dataset=True,
        note="LCSA: top-k sparse attention conditioned on clinical context"),

    "convnext_bilinear": dict(
        display="ConvNeXt-T + FT-Transformer Bilinear Fusion", family="Bilinear",
        backbone="convnext_tiny", micro_batch=32,
        origin="ours", novel_on_dataset=True,
        note="second-order outer-product cross-modal interaction"),

    "swinv2_gat": dict(
        display="Swin-V2 + Graph Attention Fusion", family="Graph",
        backbone="swinv2_cr_small_224", micro_batch=16,
        origin="ours", novel_on_dataset=True,
        note="relational graph over clinical feature nodes, GAT message passing"),

    "unet_crossattn": dict(
        display="U-Net Encoder + Metadata Cross-Attention", family="Cross-attention",
        backbone="scratch_unet", micro_batch=32,
        origin="ours", novel_on_dataset=True,
        note="multi-scale encoder trained from scratch, no ImageNet initialisation"),
}

# Non-deep reference points, trained by report.py to contextualise the table.
BASELINES = ["metadata_only_rf", "metadata_only_lr", "image_only_vit"]


def get_config(model_name: str, **overrides) -> UnifiedConfig:
    if model_name not in MODEL_ZOO:
        raise KeyError(f"{model_name!r} not in MODEL_ZOO; choose from {list(MODEL_ZOO)}")
    cfg = UnifiedConfig(model_name=model_name,
                        micro_batch=MODEL_ZOO[model_name]["micro_batch"])
    for k, v in overrides.items():
        if not hasattr(cfg, k):
            raise KeyError(f"unknown config field {k!r}")
        setattr(cfg, k, v)
    return cfg


def grad_accum_steps(cfg: UnifiedConfig) -> int:
    if cfg.effective_batch % cfg.micro_batch != 0:
        raise ValueError("effective_batch must be divisible by micro_batch")
    return cfg.effective_batch // cfg.micro_batch
'''

FILES['data.py'] = r'''"""
Unified data pipeline.

One split, one preprocessing chain, one augmentation policy -- for all models.

The split is NOT recomputed here. It is loaded from the frozen artefacts
produced by "EDA & preprocessing/final-splitted-data.ipynb" (train/val/test
zips or folders). This is what puts every architecture on the same ground.
"""
from __future__ import annotations

import os
import zipfile
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from config import (UnifiedConfig, FEATURE_SETS, ID_COLS, TARGET_COL,
                    IMAGE_COL, N_TRAIN, N_VAL, N_TEST)

Image.MAX_IMAGE_PIXELS = None


# ---------------------------------------------------------------------------
#  Split discovery / extraction
# ---------------------------------------------------------------------------
def ensure_split(data_root: str, work_dir: str = "/kaggle/working/split") -> Dict[str, str]:
    """
    Locate the frozen 70/15/15 split. Accepts either already-extracted folders
    or the train.zip / val.zip / test.zip artefacts, anywhere under data_root.
    Returns {"train": dir, "val": dir, "test": dir}.
    """
    out: Dict[str, str] = {}
    for split in ("train", "val", "test"):
        found = None
        for root, dirs, files in os.walk(data_root):
            if os.path.basename(root) == split and f"{split}.csv" in files:
                found = root
                break
        if found:
            out[split] = found
            continue
        # fall back to the zip artefacts
        zip_path = None
        for root, dirs, files in os.walk(data_root):
            if f"{split}.zip" in files:
                zip_path = os.path.join(root, f"{split}.zip")
                break
        if zip_path is None:
            raise FileNotFoundError(
                f"Could not find the frozen '{split}' split under {data_root}. "
                "Attach the splitted-data dataset to the notebook.")
        dest = os.path.join(work_dir)
        os.makedirs(dest, exist_ok=True)
        if not os.path.isdir(os.path.join(dest, split)):
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(dest)
        out[split] = os.path.join(dest, split)
    return out


def load_split(data_root: str, feature_set: str = "full"
               ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, List[str], Dict[str, str]]:
    """Load the three frozen dataframes plus the agreed metadata column list."""
    dirs = ensure_split(data_root)
    dfs = {}
    for split, d in dirs.items():
        df = pd.read_csv(os.path.join(d, f"{split}.csv"))
        dfs[split] = df

    tr, va, te = dfs["train"], dfs["val"], dfs["test"]

    # Integrity: the split must be the frozen one, otherwise results are not
    # comparable across models and the run is aborted.
    for name, df, n in (("train", tr, N_TRAIN), ("val", va, N_VAL), ("test", te, N_TEST)):
        if len(df) != n:
            raise ValueError(
                f"{name} split has {len(df)} rows but the frozen split defines {n}. "
                "Refusing to run: the benchmark requires the identical partition.")

    meta_cols = build_feature_list(tr, feature_set)
    return tr, va, te, meta_cols, dirs


def build_feature_list(df: pd.DataFrame, feature_set: str) -> List[str]:
    if feature_set not in FEATURE_SETS:
        raise KeyError(f"unknown feature_set {feature_set!r}")
    drop_prefixes = FEATURE_SETS[feature_set]["drop_prefixes"]
    cols = [c for c in df.columns if c not in ID_COLS and c != TARGET_COL]
    cols = [c for c in cols if not any(c.startswith(p) for p in drop_prefixes)]
    return sorted(cols)


# ---------------------------------------------------------------------------
#  Transforms -- IDENTICAL for every architecture
# ---------------------------------------------------------------------------
def build_transforms(cfg: UnifiedConfig, train: bool) -> transforms.Compose:
    """
    The corpus images are already DullRazor-inpainted and resized to 224x224 by
    the shared preprocessing stage, so the deterministic chain is a square
    resize (a no-op at 224) rather than a resize-and-crop, which would upscale
    and then discard the lesion border.

    Training chain adds the conservative augmentation policy described in the
    paper. Augmentations are deliberately mild so that diagnostically relevant
    colour gradients and border morphology survive.
    """
    norm = transforms.Normalize(mean=cfg.mean, std=cfg.std)
    if not train:
        return transforms.Compose([
            transforms.Resize((cfg.img_size, cfg.img_size)),
            transforms.ToTensor(),
            norm,
        ])
    return transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.RandomResizedCrop(cfg.img_size, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.5),
        transforms.RandomApply([transforms.RandomRotation(90)], p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),
        transforms.RandomAffine(degrees=0, shear=10, scale=(0.9, 1.1)),
        transforms.ToTensor(),
        norm,
        transforms.RandomErasing(p=0.3, scale=(0.01, 0.05), value="random"),
    ])


# ---------------------------------------------------------------------------
#  Dataset
# ---------------------------------------------------------------------------
class LesionDataset(Dataset):
    """Returns (image, metadata_vector, label). Metadata is already numeric."""

    def __init__(self, df: pd.DataFrame, img_dir: str, meta_cols: List[str],
                 tfm: transforms.Compose):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.meta_cols = meta_cols
        self.tfm = tfm
        self.meta = self.df[meta_cols].to_numpy(dtype=np.float32)
        self.labels = self.df[TARGET_COL].to_numpy(dtype=np.float32)
        self.files = self.df[IMAGE_COL].astype(str).tolist()

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, i: int):
        path = os.path.join(self.img_dir, self.files[i])
        with Image.open(path) as im:
            img = self.tfm(im.convert("RGB"))
        return img, torch.from_numpy(self.meta[i]), torch.tensor(self.labels[i])


def build_loaders(cfg: UnifiedConfig) -> Tuple[DataLoader, DataLoader, DataLoader, List[str]]:
    tr, va, te, meta_cols, dirs = load_split(cfg.data_root, cfg.feature_set)
    pin = torch.cuda.is_available()
    common = dict(num_workers=cfg.num_workers, pin_memory=pin,
                  persistent_workers=cfg.num_workers > 0,
                  prefetch_factor=4 if cfg.num_workers > 0 else None)

    def mk(df, split, train):
        ds = LesionDataset(df, os.path.join(dirs[split], "images"), meta_cols,
                           build_transforms(cfg, train))
        return DataLoader(ds, batch_size=cfg.micro_batch, shuffle=train,
                          drop_last=train, **common)

    return (mk(tr, "train", True), mk(va, "val", False),
            mk(te, "test", False), meta_cols)
'''

FILES['metrics.py'] = r'''"""
Unified evaluation protocol.

Every model is scored with exactly these metrics, on the same test set, with
the operating threshold chosen the same way (Youden's J on VALIDATION, never
on test) and with probabilities temperature-calibrated on VALIDATION.

Reported metrics (paper requirement):
    Accuracy, Balanced Accuracy, Precision, Recall (sensitivity), F1
plus AUC-ROC, PR-AUC, specificity, ECE and the confusion matrix.
"""
from __future__ import annotations

from typing import Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             confusion_matrix, roc_curve)


# ---------------------------------------------------------------------------
#  Threshold selection -- fitted on VALIDATION only
# ---------------------------------------------------------------------------
def choose_threshold(y_true: np.ndarray, y_prob: np.ndarray, rule: str = "youden") -> float:
    if rule == "youden":
        fpr, tpr, thr = roc_curve(y_true, y_prob)
        return float(thr[int(np.argmax(tpr - fpr))])
    if rule == "f1":
        grid = np.linspace(0.05, 0.95, 181)
        scores = [f1_score(y_true, (y_prob >= t).astype(int), average="macro") for t in grid]
        return float(grid[int(np.argmax(scores))])
    if rule == "half":
        return 0.5
    raise ValueError(f"unknown threshold rule {rule!r}")


# ---------------------------------------------------------------------------
#  Temperature scaling -- fitted on VALIDATION only
# ---------------------------------------------------------------------------
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(1))

    @property
    def temperature(self) -> float:
        return float(self.log_t.exp().item())

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / self.log_t.exp()

    def fit(self, logits: np.ndarray, labels: np.ndarray) -> float:
        z = torch.tensor(logits, dtype=torch.float32).view(-1)
        y = torch.tensor(labels, dtype=torch.float32).view(-1)
        opt = torch.optim.LBFGS([self.log_t], lr=0.05, max_iter=100)
        lossf = nn.BCEWithLogitsLoss()

        def closure():
            opt.zero_grad()
            loss = lossf(self.forward(z), y)
            loss.backward()
            return loss

        opt.step(closure)
        return self.temperature


def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 15) -> float:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.digitize(y_prob, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if not m.any():
            continue
        ece += m.mean() * abs(y_true[m].mean() - y_prob[m].mean())
    return float(ece)


# ---------------------------------------------------------------------------
#  The metric block
# ---------------------------------------------------------------------------
def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        "accuracy":          float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision":         float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":            float(recall_score(y_true, y_pred, zero_division=0)),
        "f1":                float(f1_score(y_true, y_pred, zero_division=0)),
        "f1_macro":          float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_macro":   float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro":      float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "specificity":       float(specificity),
        "auc":               float(roc_auc_score(y_true, y_prob)),
        "pr_auc":            float(average_precision_score(y_true, y_prob)),
        "ece":               expected_calibration_error(y_true, y_prob),
        "threshold":         float(threshold),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def bootstrap_ci(y_true: np.ndarray, y_prob: np.ndarray, threshold: float,
                 n: int = 1000, seed: int = 42) -> Dict[str, Tuple[float, float]]:
    """Percentile 95% CIs, so the paper can state whether gaps are meaningful."""
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    keys = ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "auc"]
    acc: Dict[str, list] = {k: [] for k in keys}
    idx_all = np.arange(len(y_true))
    for _ in range(n):
        idx = rng.choice(idx_all, size=len(idx_all), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        m = compute_metrics(y_true[idx], y_prob[idx], threshold)
        for k in keys:
            acc[k].append(m[k])
    return {k: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
            for k, v in acc.items() if v}


def delong_like_pvalue(y_true: np.ndarray, p_a: np.ndarray, p_b: np.ndarray,
                       n: int = 2000, seed: int = 42) -> float:
    """
    Bootstrap test for AUC(A) > AUC(B) on paired predictions.
    Used to state whether the top model is genuinely ahead of the runner-up.
    """
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int).ravel()
    diffs = []
    idx_all = np.arange(len(y_true))
    for _ in range(n):
        idx = rng.choice(idx_all, size=len(idx_all), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        diffs.append(roc_auc_score(y_true[idx], p_a[idx]) - roc_auc_score(y_true[idx], p_b[idx]))
    diffs = np.asarray(diffs)
    return float(2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
'''

FILES['models.py'] = r'''"""
The fourteen multimodal architectures, ported to one common interface.

Every model:
    forward(image, meta) -> logits of shape (B,)      [single logit, BCE]
    param_groups(...)    -> discriminative LR groups (backbone vs head)
    tokens/attention accessors used by xai.py

The fusion mechanisms are the ones designed in this project; only the plumbing
(interface, metadata vector, output head) has been unified so that the models
are comparable. Backbone choices and hyper-parameters now come from config.py.
"""
from __future__ import annotations

import math
import os
import warnings
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import timm
except ImportError:  # pragma: no cover
    timm = None


# ===========================================================================
#  Shared building blocks
# ===========================================================================
class TokenBackbone(nn.Module):
    """
    Wraps any image backbone into a common token interface.

    forward(x) -> (tokens, hw)
        tokens : (B, N, C)   N = 1+P for ViT-style (CLS first), P for CNNs
        hw     : (H, W) spatial grid for CNN-style backbones, else None
    """

    def __init__(self, name: str, img_size: int = 224, pretrained: bool = True,
                 drop_path: float = 0.1):
        super().__init__()
        self.name = name
        self.kind = "timm"
        self.has_cls = False
        self.actual_backbone = name

        if name == "scratch_unet":
            self.kind = "unet"
            self.net = UNetStyleEncoder(d=256)
            self.out_dim = 256
            self.hw = (4, 4)
            return

        if name.startswith("nvidia/"):
            from transformers import SegformerModel
            # nvidia/mit-b5 publishes TensorFlow-only weights on the Hub, so the
            # PyTorch encoder is taken from the fine-tuned SegFormer-B5 release
            # (identical MiT-B5 hierarchical encoder) and the decode head is
            # discarded -- we only use it as a feature extractor.
            candidates = [name,
                          "nvidia/segformer-b5-finetuned-ade-640-640",
                          "nvidia/segformer-b5-finetuned-cityscapes-1024-1024",
                          "nvidia/mit-b2"]
            last_err = None
            for cand in candidates:
                try:
                    self.net = SegformerModel.from_pretrained(cand)
                    self.actual_backbone = cand
                    break
                except Exception as exc:  # noqa: BLE001
                    last_err = exc
            else:
                raise RuntimeError(f"could not load a SegFormer encoder: {last_err}")
            self.kind = "segformer"
            self.out_dim = self.net.config.hidden_sizes[-1]
            self.hw = (img_size // 32, img_size // 32)
            return

        if name == "biovil":
            # The genuine BioViL-T checkpoint is distributed with custom modelling
            # code, so loading it requires trust_remote_code=True, i.e. executing
            # Python downloaded from the Hub. That is an explicit opt-in here
            # rather than a silent default: set UNIFIED_ALLOW_REMOTE_CODE=1 to
            # enable it. Without the opt-in we fall back to ImageNet ViT-B/16 and
            # RECORD that fact, so the paper never claims a biomedical
            # pretraining distribution the model did not actually have.
            if os.environ.get("UNIFIED_ALLOW_REMOTE_CODE") == "1":
                try:
                    from transformers import AutoModel
                    self.net = AutoModel.from_pretrained(
                        "microsoft/BiomedVLP-BioViL-T", trust_remote_code=True)
                    self.kind = "biovil"
                    self.out_dim = 768
                    self.hw = (img_size // 16, img_size // 16)
                    self.has_cls = True
                    self.actual_backbone = "microsoft/BiomedVLP-BioViL-T"
                    return
                except Exception as exc:  # noqa: BLE001
                    warnings.warn(f"BioViL-T unavailable ({exc}); "
                                  "falling back to ImageNet ViT-B/16.")
            name = "vit_base_patch16_224"
            self.actual_backbone = "vit_base_patch16_224 (BioViL-T not loaded)"

        kwargs = dict(pretrained=pretrained, num_classes=0, global_pool="",
                      drop_path_rate=drop_path)
        if "dinov2" in name:
            kwargs["img_size"] = img_size
        self.net = timm.create_model(name, **kwargs)
        self.out_dim = self.net.num_features

        with torch.no_grad():
            probe = self.net(torch.zeros(1, 3, img_size, img_size))
        if probe.ndim == 3:
            self.layout = "nlc"
            self.has_cls = probe.shape[1] % 2 == 1  # 197, 257 ... -> CLS present
            p = probe.shape[1] - (1 if self.has_cls else 0)
            side = int(round(math.sqrt(p)))
            self.hw = (side, side)
        elif probe.shape[1] == self.out_dim:
            self.layout = "nchw"
            self.hw = (probe.shape[2], probe.shape[3])
        else:
            self.layout = "nhwc"
            self.hw = (probe.shape[1], probe.shape[2])

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, Optional[Tuple[int, int]]]:
        if self.kind == "unet":
            return self.net(x), self.hw
        if self.kind == "segformer":
            out = self.net(pixel_values=x, output_hidden_states=False).last_hidden_state
            h, w = out.shape[2], out.shape[3]
            return out.flatten(2).transpose(1, 2), (h, w)
        if self.kind == "biovil":
            out = self.net(pixel_values=x).last_hidden_state   # (B, 1+P, D)
            return out, self.hw

        out = self.net(x)
        if self.layout == "nlc":
            return out, self.hw
        if self.layout == "nchw":
            return out.flatten(2).transpose(1, 2), (out.shape[2], out.shape[3])
        return out.flatten(1, 2), (out.shape[1], out.shape[2])


class ConvBNAct(nn.Module):
    def __init__(self, cin: int, cout: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.GELU())

    def forward(self, x):
        return self.block(x)


class UNetStyleEncoder(nn.Module):
    """Multi-scale encoder trained from scratch (no ImageNet initialisation)."""

    def __init__(self, d: int = 256):
        super().__init__()
        self.stem = ConvBNAct(3, 32)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), ConvBNAct(32, 64))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), ConvBNAct(64, 128))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), ConvBNAct(128, 256))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), ConvBNAct(256, 512))
        self.proj = nn.Conv2d(32 + 64 + 128 + 256 + 512, d, 1)
        self.norm = nn.LayerNorm(d)

    def forward(self, x):
        f1 = self.stem(x)
        f2 = self.down1(f1)
        f3 = self.down2(f2)
        f4 = self.down3(f3)
        f5 = self.down4(f4)
        feats = [F.adaptive_avg_pool2d(f, (4, 4)) for f in (f1, f2, f3, f4, f5)]
        feat = self.proj(torch.cat(feats, dim=1))
        return self.norm(feat.flatten(2).transpose(1, 2))


class TabTransformerEncoder(nn.Module):
    """One token per clinical feature + a CLS token. Returns (B, 1+M, D)."""

    def __init__(self, num_features: int, embed_dim: int = 256, depth: int = 2,
                 num_heads: int = 4, ff_dim: int = 512, dropout: float = 0.15):
        super().__init__()
        self.num_features = num_features
        self.embed = nn.Parameter(torch.randn(num_features, embed_dim) * 0.02)
        self.bias = nn.Parameter(torch.zeros(num_features, embed_dim))
        self.pos = nn.Parameter(torch.zeros(1, num_features, embed_dim))
        self.cls = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        nn.init.trunc_normal_(self.cls, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # vectorised feature tokenisation: (B, M) -> (B, M, D)
        tok = x.unsqueeze(-1) * self.embed.unsqueeze(0) + self.bias.unsqueeze(0)
        tok = tok + self.pos
        tok = torch.cat([self.cls.expand(x.size(0), -1, -1), tok], dim=1)
        return self.norm(self.tf(tok))


class FTTransformerEncoder(nn.Module):
    """FT-Transformer; returns the pooled CLS embedding (B, D)."""

    def __init__(self, num_features: int, d: int = 256, depth: int = 2,
                 heads: int = 4, dropout: float = 0.15):
        super().__init__()
        self.tab = TabTransformerEncoder(num_features, d, depth, heads, d * 2, dropout)

    def forward(self, x):
        return self.tab(x)[:, 0]


class MLPMetaEncoder(nn.Module):
    def __init__(self, in_dim: int, out_dim: int = 128, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, out_dim), nn.LayerNorm(out_dim), nn.GELU())

    def forward(self, x):
        return self.net(x)


class CrossAttentionBlock(nn.Module):
    """Image self-attention + metadata-queries-image cross-attention."""

    def __init__(self, dim: int, num_heads: int, ff_dim: int, dropout: float = 0.15):
        super().__init__()
        self.meta_to_img = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.img_self = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_img = nn.LayerNorm(dim)
        self.norm_meta = nn.LayerNorm(dim)
        self.ffn_img = _ffn(dim, ff_dim, dropout)
        self.ffn_meta = _ffn(dim, ff_dim, dropout)

    def forward(self, img, meta, need_attn: bool = False):
        sa, _ = self.img_self(self.norm_img(img), img, img, need_weights=False)
        img = img + sa
        img = img + self.ffn_img(img)
        ca, attn = self.meta_to_img(self.norm_meta(meta), img, img,
                                    need_weights=need_attn,
                                    average_attn_weights=False)
        meta = meta + ca
        meta = meta + self.ffn_meta(meta)
        return img, meta, attn


class BidirectionalCrossAttentionBlock(nn.Module):
    """Mutual attention: image attends to metadata and metadata to image."""

    def __init__(self, dim: int, num_heads: int, dropout: float = 0.15):
        super().__init__()
        self.img_to_meta = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.meta_to_img = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.n_img, self.n_meta = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.ffn_img, self.ffn_meta = _ffn(dim, dim * 2, dropout), _ffn(dim, dim * 2, dropout)
        self.nf_img, self.nf_meta = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, img, meta, need_attn: bool = False):
        a, attn = self.img_to_meta(img, meta, meta, need_weights=need_attn,
                                   average_attn_weights=False)
        img = self.n_img(img + self.drop(a))
        b, _ = self.meta_to_img(meta, img, img, need_weights=False)
        meta = self.n_meta(meta + self.drop(b))
        img = self.nf_img(img + self.drop(self.ffn_img(img)))
        meta = self.nf_meta(meta + self.drop(self.ffn_meta(meta)))
        return img, meta, attn


class LesionAwareCrossAttentionBlock(nn.Module):
    """
    LACA (ours). The clinical CLS token predicts a per-patch saliency gate that
    multiplicatively steers fusion toward clinically relevant lesion regions
    before cross-attention is applied. The gate is retained for XAI.
    """

    def __init__(self, dim: int, num_heads: int, ff_dim: int, n_patches: int,
                 dropout: float = 0.15):
        super().__init__()
        self.n_patches = n_patches
        self.saliency_gate = nn.Sequential(
            nn.Linear(dim, dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim // 2, n_patches))
        self.meta_to_img = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.img_self = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_img, self.norm_meta = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.ffn_img, self.ffn_meta = _ffn(dim, ff_dim, dropout), _ffn(dim, ff_dim, dropout)

    def forward(self, img, meta, need_attn: bool = False):
        gate = torch.sigmoid(self.saliency_gate(meta[:, 0]))          # (B, P)
        cls_img, patches = img[:, :1], img[:, 1:]
        p = patches.size(1)
        g = gate[:, :p] if gate.size(1) >= p else F.pad(gate, (0, p - gate.size(1)), value=1.0)
        patches = patches * g.unsqueeze(-1)
        img = torch.cat([cls_img, patches], dim=1)

        sa, _ = self.img_self(self.norm_img(img), img, img, need_weights=False)
        img = img + sa
        img = img + self.ffn_img(img)
        ca, attn = self.meta_to_img(self.norm_meta(meta), img, img,
                                    need_weights=need_attn, average_attn_weights=False)
        meta = meta + ca
        meta = meta + self.ffn_meta(meta)
        return img, meta, attn, g


class LesionConditionedSparseAttention(nn.Module):
    """
    LCSA (ours). Clinical context queries the image, but attention is restricted
    to the top-k most relevant patches per head, mimicking the way a clinician
    fixates on a small number of salient regions rather than scanning uniformly.
    """

    def __init__(self, img_dim: int, meta_dim: int, fusion_dim: int = 256,
                 n_heads: int = 8, topk: int = 16, dropout: float = 0.1):
        super().__init__()
        assert fusion_dim % n_heads == 0
        self.h, self.dh, self.topk = n_heads, fusion_dim // n_heads, topk
        self.scale = self.dh ** -0.5
        self.img_proj = nn.Linear(img_dim, fusion_dim)
        self.meta_proj = nn.Linear(meta_dim, fusion_dim)
        self.q = nn.Linear(fusion_dim, fusion_dim)
        self.k = nn.Linear(fusion_dim, fusion_dim)
        self.v = nn.Linear(fusion_dim, fusion_dim)
        self.out = nn.Linear(fusion_dim * 2, fusion_dim)
        self.norm = nn.LayerNorm(fusion_dim)
        self.drop = nn.Dropout(dropout)
        self.gate = nn.Sequential(nn.Linear(fusion_dim * 2, 2), nn.Softmax(dim=-1))
        self.last_attn: Optional[torch.Tensor] = None

    def _split(self, t):
        b, n, _ = t.shape
        return t.view(b, n, self.h, self.dh).transpose(1, 2)

    def forward(self, img_cls, patches, meta_cls, keep_attn: bool = False):
        p = self.img_proj(patches)                       # (B, P, F)
        m = self.meta_proj(meta_cls).unsqueeze(1)        # (B, 1, F)
        q, k, v = self._split(self.q(m)), self._split(self.k(p)), self._split(self.v(p))
        logits = (q @ k.transpose(-2, -1)) * self.scale  # (B, H, 1, P)

        kk = min(self.topk, logits.size(-1))
        thresh = logits.topk(kk, dim=-1).values[..., -1:]
        logits = logits.masked_fill(logits < thresh, float("-inf"))
        attn = logits.softmax(dim=-1)
        if keep_attn:
            self.last_attn = attn.detach()
        ctx = (self.drop(attn) @ v).transpose(1, 2).reshape(m.size(0), 1, -1).squeeze(1)

        img_pooled = self.img_proj(img_cls) if img_cls.dim() == 2 else p.mean(1)
        joint = torch.cat([ctx, img_pooled], dim=-1)
        w = self.gate(joint)
        fused = self.out(joint) * w[:, :1] + ctx * w[:, 1:]
        return self.norm(fused), w


class GatedFusion(nn.Module):
    """
    Per-modality sigmoid gates (ours). The gate activations are themselves an
    interpretability signal: a low image gate means the model fell back on
    clinical context for that patient.
    """

    def __init__(self, img_dim: int, meta_dim: int, fusion_dim: int, dropout: float = 0.3):
        super().__init__()
        c = img_dim + meta_dim
        self.gate_img = nn.Sequential(nn.Linear(c, c // 2), nn.GELU(),
                                      nn.Linear(c // 2, img_dim), nn.Sigmoid())
        self.gate_meta = nn.Sequential(nn.Linear(c, c // 2), nn.GELU(),
                                       nn.Linear(c // 2, meta_dim), nn.Sigmoid())
        self.img_proj = nn.Linear(img_dim, fusion_dim)
        self.meta_proj = nn.Linear(meta_dim, fusion_dim)
        self.mlp = nn.Sequential(
            nn.LayerNorm(fusion_dim), nn.Linear(fusion_dim, fusion_dim * 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(fusion_dim * 2, fusion_dim), nn.LayerNorm(fusion_dim))

    def forward(self, f_img, f_meta):
        c = torch.cat([f_img, f_meta], dim=-1)
        g_img, g_meta = self.gate_img(c), self.gate_meta(c)
        fused = self.img_proj(g_img * f_img) + self.meta_proj(g_meta * f_meta)
        return self.mlp(fused), {"gate_img": g_img.detach(), "gate_meta": g_meta.detach()}


class DenseGATv2(nn.Module):
    """
    Dense GATv2 layer over a small fully-connected node set.

    Implemented densely rather than via torch_geometric: the graph here has a
    fixed, modest node count, so dense attention is mathematically identical,
    removes a heavyweight dependency, and avoids the per-sample graph-batching
    that dominated runtime in the original implementation.
    """

    def __init__(self, dim: int, heads: int = 4, dropout: float = 0.15):
        super().__init__()
        assert dim % heads == 0
        self.h, self.dh = heads, dim // heads
        self.lin_l = nn.Linear(dim, dim)
        self.lin_r = nn.Linear(dim, dim)
        self.att = nn.Parameter(torch.empty(1, heads, 1, 1, self.dh))
        nn.init.xavier_uniform_(self.att)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):                                  # x: (B, N, D)
        b, n, _ = x.shape
        l = self.lin_l(x).view(b, n, self.h, self.dh).transpose(1, 2)   # (B,H,N,dh)
        r = self.lin_r(x).view(b, n, self.h, self.dh).transpose(1, 2)
        e = F.leaky_relu(l.unsqueeze(3) + r.unsqueeze(2), 0.2)          # (B,H,N,N,dh)
        logits = (e * self.att).sum(-1)                                 # (B,H,N,N)
        alpha = self.drop(logits.softmax(dim=-1))
        out = (alpha @ r).transpose(1, 2).reshape(b, n, -1)
        return self.norm(out + x)


def _ffn(dim: int, hidden: int, dropout: float) -> nn.Sequential:
    return nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU(),
                         nn.Dropout(dropout), nn.Linear(hidden, dim), nn.Dropout(dropout))


# ===========================================================================
#  Base class
# ===========================================================================
class FusionModel(nn.Module):
    """Common interface. Subclasses set self.backbone (if any) and self.head."""

    BACKBONE_ATTRS: Tuple[str, ...] = ("backbone", "img_encoder", "vit")

    def param_groups(self, lr_head: float, lr_backbone: float, weight_decay: float
                     ) -> List[Dict]:
        backbone_ids = set()
        for attr in self.BACKBONE_ATTRS:
            mod = getattr(self, attr, None)
            if isinstance(mod, nn.Module):
                backbone_ids |= {id(p) for p in mod.parameters()}
        bb_decay, bb_nodecay, hd_decay, hd_nodecay = [], [], [], []
        for name, p in self.named_parameters():
            if not p.requires_grad:
                continue
            no_decay = p.ndim <= 1 or name.endswith(".bias")
            is_bb = id(p) in backbone_ids
            (bb_nodecay if (is_bb and no_decay) else
             bb_decay if is_bb else
             hd_nodecay if no_decay else hd_decay).append(p)
        groups = [
            {"params": bb_decay, "lr": lr_backbone, "weight_decay": weight_decay},
            {"params": bb_nodecay, "lr": lr_backbone, "weight_decay": 0.0},
            {"params": hd_decay, "lr": lr_head, "weight_decay": weight_decay},
            {"params": hd_nodecay, "lr": lr_head, "weight_decay": 0.0},
        ]
        return [g for g in groups if g["params"]]

    def backbone_name(self) -> str:
        bb = getattr(self, "backbone", None)
        return getattr(bb, "actual_backbone", "n/a")


# ===========================================================================
#  1. ViT Early Fusion  (ours)
# ===========================================================================
class ViTEarlyFusion(FusionModel):
    """Metadata is projected into token space and prepended before block 1, so
    every transformer layer is conditioned on clinical context."""

    def __init__(self, meta_dim: int, img_size: int = 224, embed_dim: int = 768):
        super().__init__()
        self.vit = timm.create_model("vit_base_patch16_224", pretrained=True,
                                     num_classes=0, global_pool="")
        self.meta_proj = nn.Sequential(
            nn.Linear(meta_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, embed_dim), nn.LayerNorm(embed_dim))
        self.meta_pos = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.meta_pos, std=0.02)
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 256), nn.GELU(),
            nn.Dropout(0.5), nn.Linear(256, 1))
        self.last_tokens: Optional[torch.Tensor] = None

    def forward(self, image, meta):
        x = self.vit.patch_embed(image)
        x = self.vit._pos_embed(x)
        x = self.vit.patch_drop(x)
        x = self.vit.norm_pre(x)
        tok = self.meta_proj(meta).unsqueeze(1) + self.meta_pos
        x = torch.cat([x, tok], dim=1)
        x = self.vit.blocks(x)
        x = self.vit.norm(x)
        self.last_tokens = x
        return self.head(x[:, 0]).squeeze(-1)


# ===========================================================================
#  2. Hybrid ViT-MLP (late fusion)
# ===========================================================================
class HybridViTMLP(FusionModel):
    def __init__(self, meta_dim: int, img_size: int = 224, embed_dim: int = 768):
        super().__init__()
        self.vit = timm.create_model("vit_base_patch16_224", pretrained=True,
                                     num_classes=0, global_pool="token")
        self.mlp = nn.Sequential(
            nn.Linear(meta_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(128, 128), nn.GELU())
        self.head = nn.Sequential(
            nn.Linear(embed_dim + 128, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.GELU(), nn.Dropout(0.2), nn.Linear(256, 1))

    def forward(self, image, meta):
        return self.head(torch.cat([self.vit(image), self.mlp(meta)], dim=1)).squeeze(-1)


# ===========================================================================
#  3/4/8/9. Cross-attention family (backbone + TabTransformer + CA)
# ===========================================================================
class CrossAttentionFusion(FusionModel):
    """Shared implementation for the standard cross-attention family."""

    BLOCK = CrossAttentionBlock

    def __init__(self, meta_dim: int, backbone: str, img_size: int = 224,
                 d: int = 256, heads: int = 8, n_layers: int = 2, dropout: float = 0.15):
        super().__init__()
        self.backbone = TokenBackbone(backbone, img_size, drop_path=dropout)
        self.img_proj = nn.Sequential(nn.Linear(self.backbone.out_dim, d),
                                      nn.LayerNorm(d), nn.GELU())
        self.img_cls = nn.Parameter(torch.zeros(1, 1, d))
        nn.init.trunc_normal_(self.img_cls, std=0.02)
        self.meta_encoder = TabTransformerEncoder(meta_dim, d, 2, 4, d * 2, dropout)
        n_patches = self.backbone.hw[0] * self.backbone.hw[1]
        self.ca_layers = nn.ModuleList([self._make_block(d, heads, dropout, n_patches)
                                        for _ in range(n_layers)])
        self.head = nn.Sequential(
            nn.LayerNorm(d * 2), nn.Linear(d * 2, 256), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, 1))
        self.attn_maps: List[torch.Tensor] = []
        self.gates: List[torch.Tensor] = []

    def _make_block(self, d, heads, dropout, n_patches):
        return self.BLOCK(d, heads, d * 2, dropout)

    def forward(self, image, meta, keep_attn: bool = False):
        tok, _ = self.backbone(image)
        img = self.img_proj(tok)
        if not self.backbone.has_cls:
            img = torch.cat([self.img_cls.expand(img.size(0), -1, -1), img], dim=1)
        met = self.meta_encoder(meta)
        self.attn_maps, self.gates = [], []
        for blk in self.ca_layers:
            out = blk(img, met, need_attn=keep_attn)
            if len(out) == 4:
                img, met, attn, gate = out
                if keep_attn and gate is not None:
                    self.gates.append(gate.detach())
            else:
                img, met, attn = out
            if keep_attn and attn is not None:
                self.attn_maps.append(attn.detach())
        return self.head(torch.cat([img[:, 0], met[:, 0]], dim=1)).squeeze(-1)


class SegFormerLACA(CrossAttentionFusion):
    """SegFormer-B5 + Lesion-Aware Cross-Attention (ours)."""
    BLOCK = LesionAwareCrossAttentionBlock

    def _make_block(self, d, heads, dropout, n_patches):
        return LesionAwareCrossAttentionBlock(d, heads, d * 2, n_patches, dropout)


# ===========================================================================
#  5. ConvNeXt-T + Bidirectional Cross-Attention (ours)
# ===========================================================================
class ConvNeXtBiCross(FusionModel):
    def __init__(self, meta_dim: int, img_size: int = 224, d: int = 256,
                 heads: int = 8, n_layers: int = 2, meta_tokens: int = 8,
                 dropout: float = 0.15):
        super().__init__()
        self.backbone = TokenBackbone("convnext_tiny", img_size, drop_path=dropout)
        self.img_proj = nn.Sequential(nn.Linear(self.backbone.out_dim, d),
                                      nn.LayerNorm(d), nn.GELU())
        self.meta_tokens = meta_tokens
        self.meta_proj = nn.Sequential(nn.Linear(meta_dim, d * meta_tokens),
                                       nn.LayerNorm(d * meta_tokens), nn.GELU())
        self.layers = nn.ModuleList([BidirectionalCrossAttentionBlock(d, heads, dropout)
                                     for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d, d // 2), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(d // 2, 1))
        self.attn_maps: List[torch.Tensor] = []

    def forward(self, image, meta, keep_attn: bool = False):
        tok, _ = self.backbone(image)
        img = self.img_proj(tok)
        met = self.meta_proj(meta).view(meta.size(0), self.meta_tokens, -1)
        self.attn_maps = []
        for blk in self.layers:
            img, met, attn = blk(img, met, need_attn=keep_attn)
            if keep_attn and attn is not None:
                self.attn_maps.append(attn.detach())
        return self.head(img.mean(1)).squeeze(-1)


# ===========================================================================
#  6. MaxViT-T + Graph Fusion (ours)
# ===========================================================================
class GNNMaxViT(FusionModel):
    """
    Heterogeneous graph over pooled image-region nodes and metadata nodes,
    refined by GATv2 message passing.

    Image tokens are adaptively pooled to a fixed node budget so the dense
    attention stays tractable; this is what makes the graph model trainable in
    the same wall-clock envelope as the other architectures.
    """

    def __init__(self, meta_dim: int, backbone: str = "maxvit_tiny_tf_224",
                 img_size: int = 224, d: int = 256, heads: int = 4,
                 img_nodes: int = 16, meta_tokens: int = 8, dropout: float = 0.15):
        super().__init__()
        self.backbone = TokenBackbone(backbone, img_size, drop_path=dropout)
        self.img_nodes, self.meta_tokens = img_nodes, meta_tokens
        self.img_proj = nn.Sequential(nn.Linear(self.backbone.out_dim, d),
                                      nn.LayerNorm(d), nn.GELU())
        self.meta_proj = nn.Sequential(nn.Linear(meta_dim, d * meta_tokens),
                                       nn.LayerNorm(d * meta_tokens), nn.GELU())
        self.gat1 = DenseGATv2(d, heads, dropout)
        self.gat2 = DenseGATv2(d, heads, dropout)
        self.head = nn.Sequential(nn.Linear(d, d // 2), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(d // 2, 1))

    def forward(self, image, meta):
        tok, hw = self.backbone(image)
        if self.backbone.has_cls:
            tok = tok[:, 1:]
        b, n, c = tok.shape
        side = int(round(math.sqrt(self.img_nodes)))
        grid = tok.transpose(1, 2).reshape(b, c, hw[0], hw[1])
        grid = F.adaptive_avg_pool2d(grid, (side, side))
        img = self.img_proj(grid.flatten(2).transpose(1, 2))
        met = self.meta_proj(meta).view(b, self.meta_tokens, -1)
        nodes = torch.cat([img, met], dim=1)
        nodes = self.gat2(self.gat1(nodes))
        return self.head(nodes[:, :img.size(1)].mean(1)).squeeze(-1)


# ===========================================================================
#  7. DaViT-T + Gated Fusion (ours)
# ===========================================================================
class DaViTGated(FusionModel):
    def __init__(self, meta_dim: int, img_size: int = 224, fusion_dim: int = 256,
                 meta_embed: int = 128, dropout: float = 0.3):
        super().__init__()
        self.backbone = TokenBackbone("davit_tiny", img_size, drop_path=0.1)
        self.meta_encoder = MLPMetaEncoder(meta_dim, meta_embed, dropout)
        self.fusion = GatedFusion(self.backbone.out_dim, meta_embed, fusion_dim, dropout)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(fusion_dim, fusion_dim // 2), nn.GELU(),
            nn.Dropout(dropout * 0.5), nn.Linear(fusion_dim // 2, 1))
        self.last_gates: Dict[str, torch.Tensor] = {}

    def forward(self, image, meta):
        tok, _ = self.backbone(image)
        f_img = tok.mean(1)
        fused, gates = self.fusion(f_img, self.meta_encoder(meta))
        self.last_gates = gates
        return self.head(fused).squeeze(-1)


# ===========================================================================
#  10. DINOv2 + Bidirectional Cross-Attention (ours)
# ===========================================================================
class DINOv2BiCross(FusionModel):
    def __init__(self, meta_dim: int, img_size: int = 224, d: int = 256,
                 heads: int = 8, n_layers: int = 2, dropout: float = 0.15):
        super().__init__()
        self.backbone = TokenBackbone("vit_small_patch14_dinov2.lvd142m",
                                      img_size, drop_path=dropout)
        self.img_proj = nn.Sequential(nn.Linear(self.backbone.out_dim, d),
                                      nn.LayerNorm(d), nn.GELU())
        self.meta_encoder = TabTransformerEncoder(meta_dim, d, 2, 4, d * 2, dropout)
        self.layers = nn.ModuleList([BidirectionalCrossAttentionBlock(d, heads, dropout)
                                     for _ in range(n_layers)])
        self.head = nn.Sequential(
            nn.LayerNorm(d * 2), nn.Linear(d * 2, 256), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(256, 1))
        self.attn_maps: List[torch.Tensor] = []

    def forward(self, image, meta, keep_attn: bool = False):
        tok, _ = self.backbone(image)
        img = self.img_proj(tok)
        met = self.meta_encoder(meta)
        self.attn_maps = []
        for blk in self.layers:
            img, met, attn = blk(img, met, need_attn=keep_attn)
            if keep_attn and attn is not None:
                self.attn_maps.append(attn.detach())
        return self.head(torch.cat([img[:, 0], met[:, 0]], dim=1)).squeeze(-1)


# ===========================================================================
#  11. BioViL-T + LCSA (ours)
# ===========================================================================
class BioViLLCSA(FusionModel):
    def __init__(self, meta_dim: int, img_size: int = 224, d: int = 256,
                 topk: int = 16, dropout: float = 0.15):
        super().__init__()
        self.backbone = TokenBackbone("biovil", img_size, drop_path=dropout)
        self.meta_encoder = FTTransformerEncoder(meta_dim, d, 2, 4, dropout)
        self.fusion = LesionConditionedSparseAttention(
            self.backbone.out_dim, d, d, n_heads=8, topk=topk, dropout=dropout)
        self.head = nn.Sequential(
            nn.LayerNorm(d), nn.Dropout(dropout), nn.Linear(d, 128), nn.GELU(),
            nn.Dropout(dropout / 2), nn.Linear(128, 1))
        self.last_gate: Optional[torch.Tensor] = None

    def forward(self, image, meta, keep_attn: bool = False):
        tok, _ = self.backbone(image)
        cls, patches = (tok[:, 0], tok[:, 1:]) if self.backbone.has_cls else (tok.mean(1), tok)
        fused, gate = self.fusion(cls, patches, self.meta_encoder(meta), keep_attn=keep_attn)
        self.last_gate = gate.detach()
        return self.head(fused).squeeze(-1)


# ===========================================================================
#  12. ConvNeXt + FT-Transformer Bilinear Fusion (ours)
# ===========================================================================
class ConvNeXtBilinear(FusionModel):
    def __init__(self, meta_dim: int, img_size: int = 224, d: int = 256,
                 dropout: float = 0.15):
        super().__init__()
        self.backbone = TokenBackbone("convnext_tiny", img_size, drop_path=dropout)
        self.img_proj = nn.Sequential(nn.Linear(self.backbone.out_dim, d),
                                      nn.GELU(), nn.Dropout(dropout))
        self.tab = FTTransformerEncoder(meta_dim, d, 2, 4, dropout)
        self.tab_proj = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(dropout))
        self.bilinear = nn.Bilinear(d, d, d)
        self.head = nn.Sequential(nn.Linear(d * 3, d * 2), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(d * 2, 1))

    def forward(self, image, meta):
        tok, _ = self.backbone(image)
        img = self.img_proj(tok.mean(1))
        tab = self.tab_proj(self.tab(meta))
        return self.head(torch.cat([img, tab, self.bilinear(img, tab)], dim=-1)).squeeze(-1)


# ===========================================================================
#  13. Swin-V2 + Graph Attention Fusion (ours)
# ===========================================================================
class SwinV2GAT(FusionModel):
    def __init__(self, meta_dim: int, img_size: int = 224, d: int = 256,
                 heads: int = 4, dropout: float = 0.15):
        super().__init__()
        self.backbone = TokenBackbone("swinv2_cr_small_224", img_size, drop_path=dropout)
        self.img_proj = nn.Sequential(nn.Linear(self.backbone.out_dim, d),
                                      nn.GELU(), nn.Dropout(dropout))
        self.node_embed = nn.Parameter(torch.randn(meta_dim, d) * 0.02)
        self.node_bias = nn.Parameter(torch.zeros(meta_dim, d))
        self.gat1 = DenseGATv2(d, heads, dropout)
        self.gat2 = DenseGATv2(d, heads, dropout)
        self.meta_pool = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(dropout))
        self.fuse = nn.Sequential(
            nn.Linear(d * 3, d * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d * 2, d), nn.GELU(), nn.Dropout(dropout))
        self.head = nn.Linear(d, 1)

    def forward(self, image, meta):
        tok, _ = self.backbone(image)
        img = self.img_proj(tok.mean(1))
        nodes = meta.unsqueeze(-1) * self.node_embed.unsqueeze(0) + self.node_bias.unsqueeze(0)
        nodes = self.gat2(self.gat1(nodes))
        met = self.meta_pool(nodes.mean(1))
        fused = self.fuse(torch.cat([img, met, img * met], dim=-1))
        return self.head(fused).squeeze(-1)


# ===========================================================================
#  14. U-Net Encoder + Metadata Cross-Attention (ours)
# ===========================================================================
class UNetCrossAttn(FusionModel):
    def __init__(self, meta_dim: int, img_size: int = 224, d: int = 256,
                 heads: int = 8, n_layers: int = 2, dropout: float = 0.15):
        super().__init__()
        self.backbone = TokenBackbone("scratch_unet", img_size)
        self.meta_encoder = TabTransformerEncoder(meta_dim, d, 2, 4, d * 2, dropout)
        self.layers = nn.ModuleList([BidirectionalCrossAttentionBlock(d, heads, dropout)
                                     for _ in range(n_layers)])
        self.fuse = nn.Sequential(
            nn.Linear(d * 3, d * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d * 2, d), nn.GELU(), nn.Dropout(dropout))
        self.head = nn.Linear(d, 1)

    def forward(self, image, meta):
        img, _ = self.backbone(image)
        met = self.meta_encoder(meta)
        for blk in self.layers:
            img, met, _ = blk(img, met)
        i, m = img.mean(1), met.mean(1)
        return self.head(self.fuse(torch.cat([i, m, i * m], dim=-1))).squeeze(-1)


# ===========================================================================
#  Factory
# ===========================================================================
def build_model(model_name: str, meta_dim: int, img_size: int = 224) -> FusionModel:
    if model_name == "vit_early_fusion":
        return ViTEarlyFusion(meta_dim, img_size)
    if model_name == "hybrid_vit_mlp":
        return HybridViTMLP(meta_dim, img_size)
    if model_name == "convnext_bicross":
        return ConvNeXtBiCross(meta_dim, img_size)
    if model_name == "swinv2_tabtransformer":
        return CrossAttentionFusion(meta_dim, "swinv2_cr_small_224", img_size)
    if model_name == "gnn_maxvit":
        return GNNMaxViT(meta_dim, "maxvit_tiny_tf_224", img_size)
    if model_name == "segformer_laca":
        return SegFormerLACA(meta_dim, "nvidia/mit-b5", img_size)
    if model_name == "davit_gated":
        return DaViTGated(meta_dim, img_size)
    if model_name == "maxvit_crossattn":
        return CrossAttentionFusion(meta_dim, "maxvit_tiny_tf_224", img_size)
    if model_name == "effnetv2_crossattn":
        return CrossAttentionFusion(meta_dim, "tf_efficientnetv2_s", img_size)
    if model_name == "dinov2_crossattn":
        return DINOv2BiCross(meta_dim, img_size)
    if model_name == "biovil_lcsa":
        return BioViLLCSA(meta_dim, img_size)
    if model_name == "convnext_bilinear":
        return ConvNeXtBilinear(meta_dim, img_size)
    if model_name == "swinv2_gat":
        return SwinV2GAT(meta_dim, img_size)
    if model_name == "unet_crossattn":
        return UNetCrossAttn(meta_dim, img_size)
    raise KeyError(f"unknown model {model_name!r}")


def count_params(model: nn.Module) -> Tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable
'''

FILES['engine.py'] = r'''"""
Unified training / evaluation engine.

GPU throughput measures applied uniformly to every model:
  * mixed precision (fp16 autocast + GradScaler)
  * channels_last memory format for convolutional backbones
  * cudnn.benchmark autotuning
  * gradient accumulation to hold the effective batch at 32 regardless of VRAM
  * fused AdamW where available
  * non_blocking H2D copies from pinned memory, persistent workers, prefetch
  * set_to_none gradient zeroing
  * inference under no_grad + autocast
  * optional torch.compile (off by default; enable with --compile)
"""
from __future__ import annotations

import json
import math
import os
import random
import time
from dataclasses import asdict
from typing import Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from config import UnifiedConfig, grad_accum_steps
from metrics import (TemperatureScaler, bootstrap_ci, choose_threshold,
                     compute_metrics)


# ---------------------------------------------------------------------------
def sigmoid(z: np.ndarray) -> np.ndarray:
    """
    Overflow-safe logistic. The naive 1/(1+exp(-z)) overflows for |z| beyond
    about 700, which happens routinely once logits are divided by a small
    calibration temperature.
    """
    z = np.asarray(z, dtype=np.float64)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    e = np.exp(z[~pos])
    out[~pos] = e / (1.0 + e)
    return out


def seed_everything(seed: int, deterministic: bool = False) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


class ModelEMA:
    """Exponential moving average of weights; evaluated instead of the raw model."""

    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float()
                       for k, v in model.state_dict().items() if v.dtype.is_floating_point}

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)

    def copy_to(self, model: nn.Module) -> Dict[str, torch.Tensor]:
        backup = {k: v.detach().clone() for k, v in model.state_dict().items()
                  if k in self.shadow}
        sd = model.state_dict()
        for k, v in self.shadow.items():
            sd[k].copy_(v.to(sd[k].dtype))
        return backup

    @staticmethod
    def restore(model: nn.Module, backup: Dict[str, torch.Tensor]) -> None:
        sd = model.state_dict()
        for k, v in backup.items():
            sd[k].copy_(v)


class WarmupCosine:
    """Linear warmup then cosine decay, stepped per optimiser update."""

    def __init__(self, optimizer, warmup_steps: int, total_steps: int, min_lr: float):
        self.opt = optimizer
        self.warmup, self.total, self.min_lr = max(1, warmup_steps), max(2, total_steps), min_lr
        self.base = [g["lr"] for g in optimizer.param_groups]
        self.step_n = 0

    def step(self) -> None:
        self.step_n += 1
        for g, base in zip(self.opt.param_groups, self.base):
            if self.step_n <= self.warmup:
                lr = base * self.step_n / self.warmup
            else:
                p = (self.step_n - self.warmup) / max(1, self.total - self.warmup)
                lr = self.min_lr + 0.5 * (base - self.min_lr) * (1 + math.cos(math.pi * min(1.0, p)))
            g["lr"] = lr


def build_optimizer(model, cfg: UnifiedConfig):
    groups = model.param_groups(cfg.lr_head, cfg.lr_backbone, cfg.weight_decay)
    try:
        return torch.optim.AdamW(groups, fused=torch.cuda.is_available())
    except (TypeError, RuntimeError):
        return torch.optim.AdamW(groups)


# ---------------------------------------------------------------------------
@torch.no_grad()
def predict(model, loader: DataLoader, device, cfg: UnifiedConfig,
            channels_last: bool) -> Tuple[np.ndarray, np.ndarray]:
    """Returns (logits, labels) for a whole loader."""
    model.eval()
    logits, labels = [], []
    for img, meta, y in loader:
        img = img.to(device, non_blocking=True)
        if channels_last:
            img = img.contiguous(memory_format=torch.channels_last)
        meta = meta.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=cfg.amp and device.type == "cuda"):
            out = model(img, meta)
        logits.append(out.float().cpu())
        labels.append(y)
    return torch.cat(logits).numpy(), torch.cat(labels).numpy()


def train_one_model(model, loaders, cfg: UnifiedConfig, out_dir: str,
                    log=print) -> Dict:
    train_loader, val_loader, test_loader = loaders
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed_everything(cfg.seed, cfg.deterministic)
    os.makedirs(out_dir, exist_ok=True)

    channels_last = cfg.channels_last and device.type == "cuda"
    model = model.to(device)
    if channels_last:
        model = model.to(memory_format=torch.channels_last)

    accum = grad_accum_steps(cfg)
    steps_per_epoch = max(1, len(train_loader) // accum)
    opt = build_optimizer(model, cfg)
    sched = WarmupCosine(opt, cfg.warmup_epochs * steps_per_epoch,
                         cfg.epochs * steps_per_epoch, cfg.min_lr)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and device.type == "cuda")
    lossf = nn.BCEWithLogitsLoss()
    ema = ModelEMA(model, cfg.ema_decay)

    best = {"score": -1.0, "epoch": -1}
    history, patience = [], 0
    ckpt = os.path.join(out_dir, "best.pt")
    t_start = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        t0, running, nb = time.time(), 0.0, 0
        opt.zero_grad(set_to_none=True)

        for i, (img, meta, y) in enumerate(train_loader):
            img = img.to(device, non_blocking=True)
            if channels_last:
                img = img.contiguous(memory_format=torch.channels_last)
            meta = meta.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            if cfg.label_smooth > 0:
                y = y * (1 - cfg.label_smooth) + 0.5 * cfg.label_smooth

            with torch.autocast("cuda", dtype=torch.float16,
                                enabled=cfg.amp and device.type == "cuda"):
                loss = lossf(model(img, meta), y) / accum
            scaler.scale(loss).backward()
            running += loss.item() * accum
            nb += 1

            if (i + 1) % accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
                ema.update(model)

        backup = ema.copy_to(model)
        vl, vy = predict(model, val_loader, device, cfg, channels_last)
        ema.restore(model, backup)

        vp = sigmoid(vl)
        thr = choose_threshold(vy, vp, cfg.threshold_rule)
        vm = compute_metrics(vy, vp, thr)
        score = vm["auc"] if cfg.monitor == "val_auc" else vm["f1"]
        row = {"epoch": epoch, "train_loss": running / max(1, nb),
               "secs": round(time.time() - t0, 1), **{f"val_{k}": v for k, v in vm.items()}}
        history.append(row)
        log(f"  epoch {epoch:2d}/{cfg.epochs} | loss {row['train_loss']:.4f} | "
            f"val AUC {vm['auc']:.4f} | val F1 {vm['f1']:.4f} | "
            f"bal-acc {vm['balanced_accuracy']:.4f} | {row['secs']:.0f}s")

        if score > best["score"]:
            best = {"score": score, "epoch": epoch, "threshold": thr}
            torch.save({"ema": ema.shadow, "model": model.state_dict(),
                        "epoch": epoch, "cfg": asdict(cfg)}, ckpt)
            patience = 0
        else:
            patience += 1
            if patience >= cfg.early_stop_patience:
                log(f"  early stop at epoch {epoch} (best epoch {best['epoch']})")
                break

    # ---- final evaluation on the held-out test set --------------------------
    state = torch.load(ckpt, map_location=device, weights_only=False)
    model.load_state_dict(state["model"])
    ema.shadow = {k: v.to(device) for k, v in state["ema"].items()}
    ema.copy_to(model)

    val_logits, val_y = predict(model, val_loader, device, cfg, channels_last)
    test_logits, test_y = predict(model, test_loader, device, cfg, channels_last)

    temperature = 1.0
    if cfg.calibrate:
        ts = TemperatureScaler()
        temperature = ts.fit(val_logits, val_y)
        val_logits_c, test_logits_c = val_logits / temperature, test_logits / temperature
    else:
        val_logits_c, test_logits_c = val_logits, test_logits

    val_prob = sigmoid(val_logits_c)
    test_prob = sigmoid(test_logits_c)

    # threshold chosen on VALIDATION, never on test
    thr = choose_threshold(val_y, val_prob, cfg.threshold_rule)
    val_metrics = compute_metrics(val_y, val_prob, thr)
    test_metrics = compute_metrics(test_y, test_prob, thr)
    ci = bootstrap_ci(test_y, test_prob, thr, cfg.n_bootstrap, cfg.seed)

    total, trainable = (sum(p.numel() for p in model.parameters()),
                        sum(p.numel() for p in model.parameters() if p.requires_grad))

    result = {
        "model_name": cfg.model_name,
        "feature_set": cfg.feature_set,
        "backbone_loaded": model.backbone_name(),
        "config": asdict(cfg),
        "best_epoch": best["epoch"],
        "epochs_run": len(history),
        "train_minutes": round((time.time() - t_start) / 60, 1),
        "params_total": total,
        "params_trainable": trainable,
        "temperature": temperature,
        "threshold": thr,
        "val": val_metrics,
        "test": test_metrics,
        "test_ci95": ci,
        "history": history,
    }

    np.savez_compressed(os.path.join(out_dir, "predictions.npz"),
                        test_prob=test_prob, test_y=test_y,
                        val_prob=val_prob, val_y=val_y)
    with open(os.path.join(out_dir, "results.json"), "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    log(f"\n  TEST  acc {test_metrics['accuracy']:.4f} | bal-acc "
        f"{test_metrics['balanced_accuracy']:.4f} | prec {test_metrics['precision']:.4f} | "
        f"rec {test_metrics['recall']:.4f} | F1 {test_metrics['f1']:.4f} | "
        f"AUC {test_metrics['auc']:.4f}")
    return result
'''

FILES['xai.py'] = r'''"""
Unified explainability suite.

Figure policy (paper requirement): ONE representative case per figure.
The case is selected deterministically as the highest-confidence CORRECTLY
classified melanoma in the test set, so every model's XAI panel shows the same
kind of evidence and the figures are directly comparable across architectures.
The selected isic_id is written into the results so the figure caption can name
the exact case.

Methods
  image branch : Grad-CAM, Grad-CAM++, Integrated Gradients, LIME, occlusion
  token branch : attention rollout (ViT), cross-attention maps, LACA gate
  tabular      : SHAP (KernelExplainer), permutation importance, PDP
"""
from __future__ import annotations

import os
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from config import UnifiedConfig


# ---------------------------------------------------------------------------
#  Case selection -- deterministic, one case
# ---------------------------------------------------------------------------
def select_case(probs: np.ndarray, labels: np.ndarray, want_class: int = 1) -> int:
    """
    Highest-confidence correctly-classified case of `want_class`.
    Deterministic: no randomness, no cherry-picking across runs.
    """
    correct = (labels == want_class) & ((probs >= 0.5).astype(int) == want_class)
    if not correct.any():
        return int(np.argmax(probs if want_class == 1 else -probs))
    idx = np.where(correct)[0]
    return int(idx[np.argmax(probs[idx])])


def denormalize(img: torch.Tensor, mean: List[float], std: List[float]) -> np.ndarray:
    m = torch.tensor(mean).view(3, 1, 1)
    s = torch.tensor(std).view(3, 1, 1)
    out = (img.detach().cpu() * s + m).clamp(0, 1)
    return out.permute(1, 2, 0).numpy()


# ---------------------------------------------------------------------------
#  Grad-CAM / Grad-CAM++
# ---------------------------------------------------------------------------
def find_cam_layer(model: nn.Module, img: Optional[torch.Tensor] = None,
                   meta: Optional[torch.Tensor] = None,
                   min_grid: int = 7) -> Optional[nn.Module]:
    """
    Deepest convolution whose activation is still spatially informative.

    Naively taking the last Conv2d is wrong for several of these backbones: in
    the U-Net encoder it is a 1x1 projection over a 4x4 grid, which upsamples
    into meaningless blobs, and in ViT the only convolution is the patch
    embedding. The layer is therefore chosen by probing actual activation
    shapes and keeping the deepest one with a grid of at least `min_grid` per
    side, falling back to the coarsest available if none qualifies.
    """
    convs = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    if not convs:
        return None
    if img is None:
        return convs[-1]

    shapes: Dict[int, Tuple[int, int]] = {}
    hooks = []

    def mk(idx):
        def hook(_m, _i, o):
            if isinstance(o, torch.Tensor) and o.ndim == 4:
                shapes[idx] = (o.shape[2], o.shape[3])
        return hook

    for i, m in enumerate(convs):
        hooks.append(m.register_forward_hook(mk(i)))
    try:
        with torch.no_grad():
            model(img, meta)
    except Exception:  # noqa: BLE001
        return convs[-1]
    finally:
        for h in hooks:
            h.remove()

    ok = [i for i, hw in shapes.items() if min(hw) >= min_grid]
    if ok:
        return convs[max(ok)]
    return convs[max(shapes)] if shapes else convs[-1]


class GradCAM:
    """Grad-CAM and Grad-CAM++ from a single target layer."""

    def __init__(self, model: nn.Module, layer: nn.Module):
        self.model, self.layer = model, layer
        self.acts: Optional[torch.Tensor] = None
        self.grads: Optional[torch.Tensor] = None
        self._h = [layer.register_forward_hook(self._fwd),
                   layer.register_full_backward_hook(self._bwd)]

    def _fwd(self, _m, _i, o):
        self.acts = o.detach() if isinstance(o, torch.Tensor) else o[0].detach()

    def _bwd(self, _m, _gi, go):
        self.grads = go[0].detach()

    def remove(self):
        for h in self._h:
            h.remove()

    def _to_map(self, cam: torch.Tensor, size: int) -> np.ndarray:
        cam = F.relu(cam)
        cam = F.interpolate(cam[None, None], size=(size, size),
                            mode="bilinear", align_corners=False)[0, 0]
        cam = cam - cam.min()
        return (cam / (cam.max() + 1e-8)).cpu().numpy()

    def __call__(self, img: torch.Tensor, meta: torch.Tensor,
                 plus_plus: bool = False) -> np.ndarray:
        self.model.zero_grad(set_to_none=True)
        out = self.model(img, meta)
        out.sum().backward()
        a, g = self.acts, self.grads
        if a is None or g is None or a.ndim != 4:
            raise RuntimeError("target layer did not produce 4-D activations")
        if plus_plus:
            g2, g3 = g.pow(2), g.pow(3)
            denom = 2 * g2 + (a * g3).sum(dim=(2, 3), keepdim=True)
            alpha = g2 / torch.where(denom != 0, denom, torch.ones_like(denom))
            w = (alpha * F.relu(g)).sum(dim=(2, 3), keepdim=True)
        else:
            w = g.mean(dim=(2, 3), keepdim=True)
        cam = (w * a).sum(1)[0]
        return self._to_map(cam, img.shape[-1])


# ---------------------------------------------------------------------------
#  Integrated Gradients (image branch)
# ---------------------------------------------------------------------------
def integrated_gradients(model, img, meta, steps: int = 50) -> np.ndarray:
    baseline = torch.zeros_like(img)
    total = torch.zeros_like(img)
    for a in torch.linspace(0, 1, steps):
        x = (baseline + a * (img - baseline)).clone().requires_grad_(True)
        model.zero_grad(set_to_none=True)
        model(x, meta).sum().backward()
        total = total + x.grad.detach()
    attr = ((img - baseline) * total / steps)[0].abs().sum(0)
    attr = attr - attr.min()
    return (attr / (attr.max() + 1e-8)).cpu().numpy()


# ---------------------------------------------------------------------------
#  Occlusion sensitivity
# ---------------------------------------------------------------------------
@torch.no_grad()
def occlusion(model, img, meta, patch: int = 32, stride: int = 16) -> np.ndarray:
    base = torch.sigmoid(model(img, meta)).item()
    size = img.shape[-1]
    grid = np.zeros(((size - patch) // stride + 1, (size - patch) // stride + 1))
    for yi, y in enumerate(range(0, size - patch + 1, stride)):
        batch, coords = [], []
        for xi, x in enumerate(range(0, size - patch + 1, stride)):
            occ = img.clone()
            occ[0, :, y:y + patch, x:x + patch] = 0
            batch.append(occ)
            coords.append(xi)
        probs = torch.sigmoid(model(torch.cat(batch), meta.repeat(len(batch), 1)))
        for xi, p in zip(coords, probs):
            grid[yi, xi] = base - p.item()
    grid = grid - grid.min()
    grid = grid / (grid.max() + 1e-8)
    return np.array(torch.nn.functional.interpolate(
        torch.tensor(grid)[None, None].float(), size=(size, size),
        mode="bilinear", align_corners=False)[0, 0])


# ---------------------------------------------------------------------------
#  Attention rollout (ViT-style token models)
# ---------------------------------------------------------------------------
def attention_rollout(model, img, meta) -> Optional[np.ndarray]:
    """
    Abnar & Zuidema rollout over the ViT blocks.

    timm's Attention computes the attention matrix internally and returns only
    the projected output, so the matrix cannot be captured from the module's
    own forward hook. It is instead captured from `attn_drop`, which receives
    the post-softmax (B, H, N, N) matrix as its input. Fused SDPA has to be
    disabled for the duration, because in that path the matrix is never
    materialised at all.
    """
    attns: List[torch.Tensor] = []
    hooks = []
    toggled = []

    def grab(_m, inp, _o):
        a = inp[0]
        if isinstance(a, torch.Tensor) and a.ndim == 4 and a.shape[-1] == a.shape[-2]:
            attns.append(a.detach())

    for mod in model.modules():
        if mod.__class__.__name__ == "Attention" and hasattr(mod, "attn_drop"):
            if getattr(mod, "fused_attn", False):
                mod.fused_attn = False
                toggled.append(mod)
            hooks.append(mod.attn_drop.register_forward_hook(grab))

    if not hooks:
        return None
    try:
        with torch.no_grad():
            model(img, meta)
    finally:
        for h in hooks:
            h.remove()
        for m in toggled:
            m.fused_attn = True
    if not attns:
        return None
    result = torch.eye(attns[0].size(-1), device=attns[0].device)
    for a in attns:
        a = a.mean(1)[0]
        a = a + torch.eye(a.size(-1), device=a.device)
        a = a / a.sum(-1, keepdim=True)
        result = a @ result
    # Row 0 is the CLS query. Drop the CLS column, then keep only the leading
    # square block of patch tokens: early-fusion models append metadata tokens
    # after the patches, which are not part of the spatial grid.
    cls = result[0, 1:]
    side = int(cls.numel() ** 0.5)
    if side < 2:
        return None
    cls = cls[:side * side]
    m = cls.reshape(side, side)[None, None].float()
    m = F.interpolate(m, size=(img.shape[-1], img.shape[-1]),
                      mode="bilinear", align_corners=False)[0, 0]
    m = m - m.min()
    return (m / (m.max() + 1e-8)).cpu().numpy()


# ---------------------------------------------------------------------------
#  Tabular attribution
# ---------------------------------------------------------------------------
def shap_metadata(model, meta_background: np.ndarray, meta_sample: np.ndarray,
                  image_fixed: torch.Tensor, device, nsamples: int = 200
                  ) -> Optional[np.ndarray]:
    """
    SHAP over the metadata branch with the image held fixed at the selected
    case, so the attribution answers "given this lesion, what did the clinical
    fields contribute?".
    """
    try:
        import shap
    except ImportError:
        return None

    def f(x: np.ndarray) -> np.ndarray:
        out = []
        with torch.no_grad():
            for i in range(0, len(x), 32):
                chunk = torch.tensor(x[i:i + 32], dtype=torch.float32, device=device)
                img = image_fixed.repeat(len(chunk), 1, 1, 1)
                out.append(torch.sigmoid(model(img, chunk)).cpu().numpy())
        return np.concatenate(out).ravel()

    bg = shap.kmeans(meta_background, min(50, len(meta_background)))
    expl = shap.KernelExplainer(f, bg)
    return np.asarray(expl.shap_values(meta_sample, nsamples=nsamples)).ravel()


@torch.no_grad()
def permutation_importance(model, loader, device, meta_cols: List[str],
                           cfg: UnifiedConfig, n_repeats: int = 3) -> Dict[str, float]:
    """Global metadata importance: AUC drop when a column is shuffled."""
    from sklearn.metrics import roc_auc_score

    imgs, metas, ys = [], [], []
    for img, meta, y in loader:
        imgs.append(img)
        metas.append(meta)
        ys.append(y)
    img = torch.cat(imgs)
    meta = torch.cat(metas)
    y = torch.cat(ys).numpy()

    def score(mt: torch.Tensor) -> float:
        probs = []
        for i in range(0, len(img), 64):
            b = img[i:i + 64].to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16,
                                enabled=cfg.amp and device.type == "cuda"):
                probs.append(torch.sigmoid(model(b, mt[i:i + 64].to(device))).float().cpu())
        return roc_auc_score(y, torch.cat(probs).numpy())

    base = score(meta)
    rng = np.random.default_rng(cfg.seed)
    out: Dict[str, float] = {}
    for j, col in enumerate(meta_cols):
        drops = []
        for _ in range(n_repeats):
            mt = meta.clone()
            mt[:, j] = mt[rng.permutation(len(mt)), j]
            drops.append(base - score(mt))
        out[col] = float(np.mean(drops))
    return out


# ---------------------------------------------------------------------------
#  Figure rendering -- ONE case per figure
# ---------------------------------------------------------------------------
def _overlay(ax, base: np.ndarray, heat: np.ndarray, title: str, cmap="jet"):
    ax.imshow(base)
    ax.imshow(heat, cmap=cmap, alpha=0.45)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


def render_xai_panel(model, cfg: UnifiedConfig, dataset, case_idx: int,
                     device, out_dir: str, display_name: str,
                     case_id: str = "") -> Dict[str, str]:
    """
    Single-case explainability panel:
        original | Grad-CAM | Grad-CAM++ | Integrated Gradients | rollout/occlusion
    """
    os.makedirs(out_dir, exist_ok=True)
    model.eval()
    img_t, meta_t, label = dataset[case_idx]
    img = img_t[None].to(device)
    meta = meta_t[None].to(device)
    base = denormalize(img_t, cfg.mean, cfg.std)

    with torch.no_grad():
        prob = torch.sigmoid(model(img, meta)).item()

    maps: Dict[str, np.ndarray] = {}
    layer = find_cam_layer(model, img, meta)
    if layer is not None:
        cam = GradCAM(model, layer)
        try:
            maps["Grad-CAM"] = cam(img, meta, plus_plus=False)
            maps["Grad-CAM++"] = cam(img, meta, plus_plus=True)
        except Exception:
            pass
        finally:
            cam.remove()

    try:
        maps["Integrated Gradients"] = integrated_gradients(model, img, meta, cfg.ig_steps)
    except Exception:
        pass

    roll = attention_rollout(model, img, meta)
    if roll is not None:
        maps["Attention Rollout"] = roll
    elif "Grad-CAM" not in maps:
        try:
            maps["Occlusion"] = occlusion(model, img, meta)
        except Exception:
            pass

    n = 1 + len(maps)
    fig, axes = plt.subplots(1, n, figsize=(3.1 * n, 3.4))
    axes = np.atleast_1d(axes)
    axes[0].imshow(base)
    truth = "melanoma" if float(label) == 1 else "non-melanoma"
    axes[0].set_title(f"{case_id or 'test case'}\ntrue: {truth} | p={prob:.3f}", fontsize=9)
    axes[0].axis("off")
    for ax, (name, heat) in zip(axes[1:], maps.items()):
        _overlay(ax, base, heat, name)
    fig.suptitle(f"{display_name} - single-case explanation", fontsize=11)
    fig.tight_layout()
    path = os.path.join(out_dir, "xai_panel.png")
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return {"panel": path, "prob": prob, "label": float(label), "methods": list(maps)}


def render_shap_bar(values: np.ndarray, meta_cols: List[str], out_dir: str,
                    display_name: str, top_k: int = 10) -> str:
    os.makedirs(out_dir, exist_ok=True)
    order = np.argsort(np.abs(values))[::-1][:top_k][::-1]
    fig, ax = plt.subplots(figsize=(7.2, 0.42 * len(order) + 1.4))
    colors = ["#c0392b" if values[i] > 0 else "#2471a3" for i in order]
    ax.barh(range(len(order)), values[order], color=colors)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([meta_cols[i] for i in order], fontsize=8)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_xlabel("SHAP value (impact on melanoma probability)", fontsize=9)
    ax.set_title(f"{display_name} - metadata attribution (single case)", fontsize=10)
    fig.tight_layout()
    path = os.path.join(out_dir, "shap_metadata.png")
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return path
'''

FILES['tune.py'] = r'''"""
Optuna hyper-parameter search -- deliberately SHARED and EQUAL-BUDGET.

Fairness rule enforced here: the search space and the trial budget are
identical for every architecture, and the objective is validation AUC only.
The test set is never touched during tuning. If tuning is used for one model
in the benchmark it must be used for all of them with the same budget;
otherwise the comparison silently rewards whichever model received more
search effort, which is one of the defects this rewrite exists to remove.

    python tune.py --model vit_early_fusion --trials 15
"""
from __future__ import annotations

import argparse
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import numpy as np
import torch

from config import MODEL_ZOO, get_config
from data import build_loaders
from engine import (ModelEMA, WarmupCosine, build_optimizer, predict, sigmoid,
                    seed_everything)
from metrics import choose_threshold, compute_metrics
from models import build_model

# Shared search space -- identical for all 14 architectures.
SEARCH_SPACE = {
    "lr_head":      ("loguniform", 5e-5, 1e-3),
    "lr_backbone":  ("loguniform", 5e-6, 1e-4),
    "weight_decay": ("loguniform", 1e-3, 1e-1),
    "label_smooth": ("categorical", [0.0, 0.05, 0.1]),
    "ema_decay":    ("categorical", [0.995, 0.999, 0.9995]),
}


def _suggest(trial, name, spec):
    kind = spec[0]
    if kind == "loguniform":
        return trial.suggest_float(name, spec[1], spec[2], log=True)
    if kind == "uniform":
        return trial.suggest_float(name, spec[1], spec[2])
    if kind == "categorical":
        return trial.suggest_categorical(name, spec[1])
    raise ValueError(kind)


def _short_train(model, loaders, cfg, epochs: int) -> float:
    """Truncated training run scored on validation AUC."""
    train_loader, val_loader, _ = loaders
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    channels_last = cfg.channels_last and device.type == "cuda"
    if channels_last:
        model = model.to(memory_format=torch.channels_last)

    accum = max(1, cfg.effective_batch // cfg.micro_batch)
    steps = max(1, len(train_loader) // accum)
    opt = build_optimizer(model, cfg)
    sched = WarmupCosine(opt, steps, epochs * steps, cfg.min_lr)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and device.type == "cuda")
    lossf = torch.nn.BCEWithLogitsLoss()
    ema = ModelEMA(model, cfg.ema_decay)

    best = 0.0
    for _ in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)
        for i, (img, meta, y) in enumerate(train_loader):
            img = img.to(device, non_blocking=True)
            if channels_last:
                img = img.contiguous(memory_format=torch.channels_last)
            meta, y = meta.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if cfg.label_smooth:
                y = y * (1 - cfg.label_smooth) + 0.5 * cfg.label_smooth
            with torch.autocast("cuda", dtype=torch.float16,
                                enabled=cfg.amp and device.type == "cuda"):
                loss = lossf(model(img, meta), y) / accum
            scaler.scale(loss).backward()
            if (i + 1) % accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
                ema.update(model)
        backup = ema.copy_to(model)
        logits, y_true = predict(model, val_loader, device, cfg, channels_last)
        ema.restore(model, backup)
        prob = sigmoid(logits)
        thr = choose_threshold(y_true, prob, cfg.threshold_rule)
        best = max(best, compute_metrics(y_true, prob, thr)["auc"])
    return best


def tune_model(model_name: str, n_trials: int = 15, epochs: int = 4,
               data_root: str = "/kaggle/input",
               out_root: str = "/kaggle/working/outputs") -> dict:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    out_dir = os.path.join(out_root, f"{model_name}__tuning")
    os.makedirs(out_dir, exist_ok=True)

    def objective(trial):
        params = {k: _suggest(trial, k, v) for k, v in SEARCH_SPACE.items()}
        cfg = get_config(model_name, data_root=data_root, num_workers=4, **params)
        seed_everything(cfg.seed, cfg.deterministic)
        tr, va, te, meta_cols = build_loaders(cfg)
        model = build_model(model_name, len(meta_cols), cfg.img_size)
        try:
            return _short_train(model, (tr, va, te), cfg, epochs)
        finally:
            del model
            torch.cuda.empty_cache()

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=1))
    study.optimize(objective, n_trials=n_trials, gc_after_trial=True)

    out = {"model": model_name, "n_trials": n_trials, "tuning_epochs": epochs,
           "search_space": {k: list(v) for k, v in SEARCH_SPACE.items()},
           "best_value_val_auc": study.best_value, "best_params": study.best_params,
           "trials": [{"number": t.number, "value": t.value, "params": t.params}
                      for t in study.trials]}
    with open(os.path.join(out_dir, "optuna.json"), "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)
    print(f"  best val AUC {study.best_value:.4f} with {study.best_params}")
    return out


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, choices=list(MODEL_ZOO))
    ap.add_argument("--trials", type=int, default=15)
    ap.add_argument("--epochs", type=int, default=4)
    ap.add_argument("--data-root", default="/kaggle/input")
    ap.add_argument("--out-root", default="/kaggle/working/outputs")
    a = ap.parse_args()
    tune_model(a.model, a.trials, a.epochs, a.data_root, a.out_root)
'''

FILES['report.py'] = r'''"""
Aggregates every finished run into the paper's tables and figures.

    python report.py --out-root outputs --paper-dir ../paper/generated

Produces
    results_all.csv                 every run, every metric
    table_main.tex                  all models ranked by AUC (IEEEtran)
    table_ablation.tex              feature-drop ablation
    table_highlight.tex             which fusion designs are ours / novel here
    fig_roc_top3.png                ROC curves, three best by AUC
    fig_confusion_top3.png          confusion matrices, three best by AUC
    fig_auc_ranking.png             all models ranked, coloured by family
    baselines.json                  metadata-only and image-only reference points
"""
from __future__ import annotations

import argparse
import glob
import json
import os
from typing import Dict, List

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, confusion_matrix

from config import MODEL_ZOO
from metrics import delong_like_pvalue

FAMILY_COLORS = {
    "Early fusion": "#1f77b4", "Late fusion": "#7f9fc4",
    "Cross-attention": "#d62728", "Sparse cross-attention": "#e377c2",
    "Gated": "#2ca02c", "Graph": "#9467bd", "Bilinear": "#ff7f0e",
}


def load_runs(out_root: str) -> List[Dict]:
    runs = []
    for path in sorted(glob.glob(os.path.join(out_root, "*", "results.json"))):
        with open(path, encoding="utf-8") as f:
            r = json.load(f)
        r["_dir"] = os.path.dirname(path)
        runs.append(r)
    return runs


def runs_to_frame(runs: List[Dict]) -> pd.DataFrame:
    rows = []
    for r in runs:
        t = r["test"]
        ci = r.get("test_ci95", {})
        zoo = MODEL_ZOO.get(r["model_name"], {})
        rows.append({
            "model": r["model_name"],
            "display": r.get("display", zoo.get("display", r["model_name"])),
            "family": r.get("family", zoo.get("family", "")),
            "origin": r.get("origin", zoo.get("origin", "")),
            "novel_on_dataset": r.get("novel_on_dataset", zoo.get("novel_on_dataset", False)),
            "feature_set": r["feature_set"],
            "accuracy": t["accuracy"], "balanced_accuracy": t["balanced_accuracy"],
            "precision": t["precision"], "recall": t["recall"], "f1": t["f1"],
            "f1_macro": t["f1_macro"], "specificity": t["specificity"],
            "auc": t["auc"], "pr_auc": t["pr_auc"], "ece": t["ece"],
            "auc_lo": ci.get("auc", (np.nan, np.nan))[0],
            "auc_hi": ci.get("auc", (np.nan, np.nan))[1],
            "threshold": t["threshold"], "temperature": r.get("temperature", 1.0),
            "params_m": r.get("params_total", 0) / 1e6,
            "epochs_run": r.get("epochs_run", 0),
            "best_epoch": r.get("best_epoch", 0),
            "train_minutes": r.get("train_minutes", 0),
            "backbone_loaded": r.get("backbone_loaded", ""),
            "tn": t["tn"], "fp": t["fp"], "fn": t["fn"], "tp": t["tp"],
            "_dir": r["_dir"],
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
#  Reference baselines
# ---------------------------------------------------------------------------
def compute_baselines(data_root: str) -> Dict:
    """
    Metadata-only references. These bound how much of the headline score is
    attributable to clinical fields rather than to the dermoscopic image, and
    they are the reason the leakage ablation exists.
    """
    import sys
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    from data import load_split, build_feature_list
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from metrics import compute_metrics, choose_threshold

    tr, va, te, _, _ = load_split(data_root, "full")
    out: Dict[str, Dict] = {}
    for fs in ("full", "no_confirm", "no_age", "no_both"):
        cols = build_feature_list(tr, fs)
        for name, mk in (("rf", lambda: RandomForestClassifier(
                              n_estimators=400, random_state=42, n_jobs=-1,
                              class_weight="balanced")),
                         ("logreg", lambda: LogisticRegression(
                              max_iter=2000, class_weight="balanced"))):
            m = mk().fit(tr[cols], tr["class"])
            vp = m.predict_proba(va[cols])[:, 1]
            tp = m.predict_proba(te[cols])[:, 1]
            thr = choose_threshold(va["class"].to_numpy(), vp)
            out[f"metadata_only_{name}__{fs}"] = {
                "n_features": len(cols),
                **compute_metrics(te["class"].to_numpy(), tp, thr)}

    # the single-flag shortcut rule
    if "diagnosis_confirm_type_histopathology" in te.columns:
        rule = te["diagnosis_confirm_type_histopathology"].to_numpy().astype(float)
        out["shortcut_rule_histopathology"] = compute_metrics(
            te["class"].to_numpy(), rule, 0.5)
        nonhisto = te[te["diagnosis_confirm_type_histopathology"] == 0]
        out["shortcut_rule_histopathology"]["melanoma_among_non_histopathology"] = \
            int(nonhisto["class"].sum())
        out["shortcut_rule_histopathology"]["n_non_histopathology"] = int(len(nonhisto))
    return out


# ---------------------------------------------------------------------------
#  Figures
# ---------------------------------------------------------------------------
def fig_roc_top3(df: pd.DataFrame, out_path: str) -> List[str]:
    top = df.nlargest(3, "auc")
    fig, ax = plt.subplots(figsize=(5.4, 5.0))
    for _, row in top.iterrows():
        p = np.load(os.path.join(row["_dir"], "predictions.npz"))
        fpr, tpr, _ = roc_curve(p["test_y"], p["test_prob"])
        ax.plot(fpr, tpr, lw=2,
                label=f"{row['display']} (AUC {row['auc']:.4f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="chance")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("ROC - three best architectures (held-out test set)")
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return top["display"].tolist()


def fig_confusion_top3(df: pd.DataFrame, out_path: str) -> None:
    top = df.nlargest(3, "auc")
    fig, axes = plt.subplots(1, 3, figsize=(12.2, 4.0))
    for ax, (_, row) in zip(np.atleast_1d(axes), top.iterrows()):
        cm = np.array([[row["tn"], row["fp"]], [row["fn"], row["tp"]]])
        cmn = cm / cm.sum(axis=1, keepdims=True)
        im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{cm[i, j]}\n({cmn[i, j]*100:.1f}%)",
                        ha="center", va="center", fontsize=10,
                        color="white" if cmn[i, j] > 0.55 else "black")
        ax.set_xticks([0, 1], ["pred. benign", "pred. melanoma"], fontsize=8)
        ax.set_yticks([0, 1], ["true benign", "true melanoma"], fontsize=8)
        ax.set_title(f"{row['display']}\nAUC {row['auc']:.4f} | F1 {row['f1']:.4f}",
                     fontsize=9)
    fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
    fig.suptitle("Confusion matrices - three best architectures", fontsize=11)
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


def fig_auc_ranking(df: pd.DataFrame, out_path: str, baselines: Dict | None = None) -> None:
    d = df.sort_values("auc")
    fig, ax = plt.subplots(figsize=(8.4, 0.42 * len(d) + 1.8))
    colors = [FAMILY_COLORS.get(f, "#888888") for f in d["family"]]
    err = None
    if d["auc_lo"].notna().all():
        err = np.vstack([d["auc"] - d["auc_lo"], d["auc_hi"] - d["auc"]])
    ax.barh(range(len(d)), d["auc"], color=colors, xerr=err,
            error_kw=dict(lw=0.8, capsize=2, ecolor="#333333"))
    ax.set_yticks(range(len(d)))
    ax.set_yticklabels(d["display"], fontsize=8)
    ax.set_xlabel("Test AUC-ROC (bars show bootstrap 95% CI)")
    lo = max(0.5, float(d["auc"].min()) - 0.05)
    ax.set_xlim(lo, 1.0)
    if baselines:
        key = "metadata_only_rf__full"
        if key in baselines:
            ax.axvline(baselines[key]["auc"], color="k", ls="--", lw=1.2)
            ax.text(baselines[key]["auc"], len(d) - 0.4,
                    f"  metadata-only RF ({baselines[key]['auc']:.3f})",
                    fontsize=7.5, va="top")
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in FAMILY_COLORS.values()]
    ax.legend(handles, FAMILY_COLORS.keys(), fontsize=7, loc="lower right")
    ax.set_title("Architecture ranking by test AUC", fontsize=11)
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------------
#  LaTeX tables
# ---------------------------------------------------------------------------
def _esc(s: str) -> str:
    return str(s).replace("&", r"\&").replace("_", r"\_").replace("%", r"\%")


def table_main(df: pd.DataFrame, path: str) -> None:
    d = df.sort_values("auc", ascending=False).reset_index(drop=True)
    lines = [
        r"\begin{table*}[t]", r"\centering",
        r"\caption{Test-set performance of all %d multimodal architectures, "
        r"ranked by AUC-ROC. All models share one 70/15/15 split, one "
        r"preprocessing pipeline, and one training protocol "
        r"(Section~\ref{sec:setup}). Operating thresholds were selected on the "
        r"validation split by Youden's $J$ and applied unchanged to the test "
        r"split. $\dagger$ marks fusion mechanisms designed in this work."
        % len(d),
        r"\label{tab:main}}",
        r"\begin{tabular}{clccccccc}", r"\toprule",
        r"Rank & Model & Acc. & Bal.\,Acc. & Prec. & Rec. & F1 & AUC (95\% CI) & Fusion \\",
        r"\midrule",
    ]
    for i, r in d.iterrows():
        mark = r"$\dagger$" if r["origin"] == "ours" else ""
        ci = (f"{r['auc']:.4f} \\scriptsize{{({r['auc_lo']:.3f}--{r['auc_hi']:.3f})}}"
              if np.isfinite(r["auc_lo"]) else f"{r['auc']:.4f}")
        lines.append(
            f"{i+1} & {_esc(r['display'])}{mark} & {r['accuracy']:.4f} & "
            f"{r['balanced_accuracy']:.4f} & {r['precision']:.4f} & "
            f"{r['recall']:.4f} & {r['f1']:.4f} & {ci} & {_esc(r['family'])} \\\\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")


def table_ablation(df: pd.DataFrame, path: str, baselines: Dict | None = None) -> None:
    piv = df.pivot_table(index="display", columns="feature_set", values="auc")
    order = [c for c in ("full", "no_confirm", "no_age", "no_both") if c in piv.columns]
    piv = piv[order].sort_values(order[0], ascending=False)
    head = " & ".join(_esc(c.replace("_", "-")) for c in order)
    lines = [
        r"\begin{table}[t]", r"\centering",
        r"\caption{Metadata ablation (test AUC). \textit{no-confirm} removes the "
        r"diagnosis-confirmation field, which deterministically encodes the "
        r"label for roughly a quarter of the corpus; \textit{no-age} removes "
        r"the age scalar and age bands.}", r"\label{tab:ablation}",
        r"\begin{tabular}{l" + "c" * len(order) + "}", r"\toprule",
        f"Model & {head} \\\\", r"\midrule",
    ]
    for name, row in piv.iterrows():
        vals = " & ".join("--" if pd.isna(v) else f"{v:.4f}" for v in row)
        lines.append(f"{_esc(name)} & {vals} \\\\")
    if baselines:
        lines.append(r"\midrule")
        vals = []
        for c in order:
            k = f"metadata_only_rf__{c}"
            vals.append(f"{baselines[k]['auc']:.4f}" if k in baselines else "--")
        lines.append("Metadata-only RF (no image) & " + " & ".join(vals) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")


def table_highlight(df: pd.DataFrame, path: str) -> None:
    d = df[df["feature_set"] == "full"].sort_values("auc", ascending=False)
    lines = [
        r"\begin{table}[t]", r"\centering",
        r"\caption{Provenance of each configuration. \emph{Designed here} marks "
        r"fusion mechanisms introduced by this work; \emph{new on this corpus} "
        r"marks backbones not previously evaluated on ISIC-DICM-17K.}",
        r"\label{tab:provenance}",
        r"\begin{tabular}{lccl}", r"\toprule",
        r"Model & Designed here & New on this corpus & Mechanism \\", r"\midrule",
    ]
    for _, r in d.iterrows():
        note = MODEL_ZOO.get(r["model"], {}).get("note", "")
        lines.append(
            f"{_esc(r['display'])} & {'yes' if r['origin']=='ours' else 'no'} & "
            f"{'yes' if r['novel_on_dataset'] else 'no'} & {_esc(note)} \\\\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")


# ---------------------------------------------------------------------------
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out-root", default="outputs")
    ap.add_argument("--paper-dir", default="../paper/generated")
    ap.add_argument("--data-root", default=None,
                    help="if given, also fit the metadata-only baselines")
    a = ap.parse_args()
    os.makedirs(a.paper_dir, exist_ok=True)

    runs = load_runs(a.out_root)
    if not runs:
        raise SystemExit(f"no results.json found under {a.out_root}")
    df = runs_to_frame(runs)
    df.to_csv(os.path.join(a.paper_dir, "results_all.csv"), index=False)
    print(f"loaded {len(df)} runs ({df['model'].nunique()} distinct models)")

    baselines = None
    if a.data_root:
        baselines = compute_baselines(a.data_root)
        with open(os.path.join(a.paper_dir, "baselines.json"), "w", encoding="utf-8") as f:
            json.dump(baselines, f, indent=2)
        k = "metadata_only_rf__full"
        print(f"metadata-only RF (no image): AUC {baselines[k]['auc']:.4f}")

    main_df = df[df["feature_set"] == "full"].copy()
    if main_df.empty:
        main_df = df.copy()

    table_main(main_df, os.path.join(a.paper_dir, "table_main.tex"))
    table_highlight(df, os.path.join(a.paper_dir, "table_highlight.tex"))
    if df["feature_set"].nunique() > 1:
        table_ablation(df, os.path.join(a.paper_dir, "table_ablation.tex"), baselines)

    top = fig_roc_top3(main_df, os.path.join(a.paper_dir, "fig_roc_top3.png"))
    fig_confusion_top3(main_df, os.path.join(a.paper_dir, "fig_confusion_top3.png"))
    fig_auc_ranking(main_df, os.path.join(a.paper_dir, "fig_auc_ranking.png"), baselines)
    print("top-3 by AUC:", ", ".join(top))

    # is the winner genuinely ahead of the runner-up?
    best2 = main_df.nlargest(2, "auc")
    if len(best2) == 2:
        pa = np.load(os.path.join(best2.iloc[0]["_dir"], "predictions.npz"))
        pb = np.load(os.path.join(best2.iloc[1]["_dir"], "predictions.npz"))
        p = delong_like_pvalue(pa["test_y"], pa["test_prob"], pb["test_prob"])
        print(f"AUC difference {best2.iloc[0]['display']} vs "
              f"{best2.iloc[1]['display']}: bootstrap p = {p:.4f}")
        with open(os.path.join(a.paper_dir, "significance.json"), "w",
                  encoding="utf-8") as f:
            json.dump({"model_a": best2.iloc[0]["display"],
                       "model_b": best2.iloc[1]["display"],
                       "auc_a": float(best2.iloc[0]["auc"]),
                       "auc_b": float(best2.iloc[1]["auc"]),
                       "bootstrap_p": p}, f, indent=2)
    print(f"wrote tables and figures to {a.paper_dir}")


if __name__ == "__main__":
    main()
'''

FILES['run.py'] = r'''"""
Single entry point for every experiment in the study.

    python run.py --model vit_early_fusion
    python run.py --model segformer_laca --feature-set no_confirm
    python run.py --all                       # sequential sweep
    python run.py --model davit_gated --tune 20   # Optuna search first

Writes, per run:
    outputs/<model>__<feature_set>/results.json     metrics + config + history
    outputs/<model>__<feature_set>/predictions.npz  test/val probabilities
    outputs/<model>__<feature_set>/best.pt          checkpoint
    outputs/<model>__<feature_set>/xai_panel.png    single-case XAI figure
    outputs/<model>__<feature_set>/shap_metadata.png
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
import warnings

import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from config import MODEL_ZOO, FEATURE_SETS, get_config
from data import build_loaders
from engine import seed_everything, train_one_model
from models import build_model, count_params
import xai as X

warnings.filterwarnings("ignore", category=UserWarning)


def run_one(model_name: str, feature_set: str, args) -> dict:
    cfg = get_config(model_name, feature_set=feature_set,
                     data_root=args.data_root, out_root=args.out_root,
                     epochs=args.epochs, num_workers=args.workers)
    if args.micro_batch:
        cfg.micro_batch = args.micro_batch

    tag = f"{model_name}__{feature_set}"
    out_dir = os.path.join(cfg.out_root, tag)
    os.makedirs(out_dir, exist_ok=True)

    print("=" * 78)
    print(f"  {MODEL_ZOO[model_name]['display']}   [{feature_set}]")
    print(f"  {MODEL_ZOO[model_name]['note']}")
    print("=" * 78)

    seed_everything(cfg.seed, cfg.deterministic)
    train_loader, val_loader, test_loader, meta_cols = build_loaders(cfg)
    print(f"  split  train={len(train_loader.dataset)} val={len(val_loader.dataset)} "
          f"test={len(test_loader.dataset)} | metadata features={len(meta_cols)}")

    model = build_model(model_name, len(meta_cols), cfg.img_size)
    total, trainable = count_params(model)
    print(f"  params {total/1e6:.1f}M ({trainable/1e6:.1f}M trainable) | "
          f"micro-batch {cfg.micro_batch} x accum {cfg.effective_batch // cfg.micro_batch} "
          f"= effective {cfg.effective_batch}\n")

    result = train_one_model(model, (train_loader, val_loader, test_loader),
                             cfg, out_dir)
    result["meta_cols"] = meta_cols
    result["display"] = MODEL_ZOO[model_name]["display"]
    result["family"] = MODEL_ZOO[model_name]["family"]
    result["origin"] = MODEL_ZOO[model_name]["origin"]
    result["novel_on_dataset"] = MODEL_ZOO[model_name]["novel_on_dataset"]

    # ---- single-case XAI -------------------------------------------------
    if not args.no_xai:
        try:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            preds = np.load(os.path.join(out_dir, "predictions.npz"))
            case = X.select_case(preds["test_prob"], preds["test_y"], want_class=1)
            ds = test_loader.dataset
            case_id = str(ds.df.iloc[case].get("isic_id", f"idx{case}"))
            print(f"\n  XAI on a single case: {case_id} "
                  f"(p={preds['test_prob'][case]:.3f})")

            panel = X.render_xai_panel(model, cfg, ds, case, device, out_dir,
                                       result["display"], case_id)
            result["xai"] = {"case_index": int(case), "case_id": case_id, **
                             {k: v for k, v in panel.items() if k != "panel"}}
            print(f"    panel -> {panel['panel']}  ({', '.join(panel['methods'])})")

            bg = ds.meta[np.random.default_rng(cfg.seed)
                         .choice(len(ds), min(cfg.shap_background, len(ds)), replace=False)]
            img_fixed = ds[case][0][None].to(device)
            sv = X.shap_metadata(model, bg, ds.meta[case], img_fixed, device,
                                 cfg.shap_nsamples)
            if sv is not None:
                p = X.render_shap_bar(sv, meta_cols, out_dir, result["display"])
                result["xai"]["shap_values"] = {c: float(v) for c, v in zip(meta_cols, sv)}
                print(f"    shap  -> {p}")
        except Exception as exc:  # noqa: BLE001
            print(f"    [warn] XAI stage failed: {type(exc).__name__}: {exc}")
            result["xai_error"] = f"{type(exc).__name__}: {exc}"

    with open(os.path.join(out_dir, "results.json"), "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)
    return result


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default=None, choices=list(MODEL_ZOO))
    ap.add_argument("--all", action="store_true", help="run every model in the zoo")
    ap.add_argument("--feature-set", default="full", choices=list(FEATURE_SETS))
    ap.add_argument("--ablation", action="store_true",
                    help="also run no_confirm / no_age / no_both for the model")
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--micro-batch", type=int, default=0)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--data-root", default="/kaggle/input")
    ap.add_argument("--out-root", default="/kaggle/working/outputs")
    ap.add_argument("--tune", type=int, default=0, help="Optuna trials before the final run")
    ap.add_argument("--no-xai", action="store_true")
    args = ap.parse_args()

    if not args.model and not args.all:
        ap.error("pass --model NAME or --all")

    models = list(MODEL_ZOO) if args.all else [args.model]
    feature_sets = ([args.feature_set] if not args.ablation
                    else [args.feature_set] + [f for f in FEATURE_SETS if f != args.feature_set])

    if args.tune:
        from tune import tune_model
        for m in models:
            tune_model(m, n_trials=args.tune, data_root=args.data_root,
                       out_root=args.out_root)

    summary = []
    for m in models:
        for fs in feature_sets:
            t0 = time.time()
            try:
                r = run_one(m, fs, args)
                summary.append({"model": m, "feature_set": fs,
                                "auc": r["test"]["auc"], "f1": r["test"]["f1"],
                                "acc": r["test"]["accuracy"],
                                "minutes": round((time.time() - t0) / 60, 1)})
            except Exception as exc:  # noqa: BLE001
                import traceback
                traceback.print_exc()
                summary.append({"model": m, "feature_set": fs, "error": str(exc)})

    print("\n" + "=" * 78)
    print("  SWEEP SUMMARY")
    print("=" * 78)
    for s in summary:
        if "error" in s:
            print(f"  {s['model']:24s} {s['feature_set']:11s} FAILED: {s['error'][:60]}")
        else:
            print(f"  {s['model']:24s} {s['feature_set']:11s} "
                  f"AUC {s['auc']:.4f}  F1 {s['f1']:.4f}  acc {s['acc']:.4f}  "
                  f"({s['minutes']} min)")
    os.makedirs(args.out_root, exist_ok=True)
    with open(os.path.join(args.out_root, "sweep_summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)


if __name__ == "__main__":
    main()
'''

for name, src in FILES.items():
    p = pathlib.Path("/kaggle/working/unified") / name
    p.write_text(src, encoding="utf-8")
print("wrote", len(FILES), "modules to /kaggle/working/unified")

## 3. Locate the frozen split

In [ ]:
# Locate the frozen 70/15/15 split that every model in this study shares.
import os, glob
CANDIDATES = glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*")
print("attached inputs:")
for c in sorted(set(CANDIDATES))[:40]:
    print("  ", c)

DATA_ROOT = "/kaggle/input"
print("\nDATA_ROOT =", DATA_ROOT)


## 4. Train and evaluate

In [ ]:
# ---------------------------------------------------------------------------
#  DaViT-T + Gated Fusion
#  per-modality sigmoid gates, directly inspectable as an XAI signal
# ---------------------------------------------------------------------------
import sys, os
sys.path.insert(0, "/kaggle/working/unified")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MODEL       = "davit_gated"
FEATURE_SET = "full"      # set to "no_confirm" / "no_age" / "no_both" for the ablation
EPOCHS      = 20

import argparse
from run import run_one

args = argparse.Namespace(
    data_root="/kaggle/input", out_root="/kaggle/working/outputs",
    epochs=EPOCHS, micro_batch=0, workers=2, no_xai=False, tune=0)

result = run_one(MODEL, FEATURE_SET, args)


## 5. Results

In [ ]:
# Final metric block -- exactly the metrics the paper reports.
import json, glob, os
for p in sorted(glob.glob("/kaggle/working/outputs/*/results.json")):
    r = json.load(open(p))
    t = r["test"]
    print(f"\n{r.get('display', r['model_name'])}  [{r['feature_set']}]")
    print(f"  backbone loaded : {r.get('backbone_loaded','n/a')}")
    print(f"  best epoch      : {r['best_epoch']} / {r['epochs_run']}  "
          f"({r.get('train_minutes','?')} min)")
    print(f"  Accuracy        : {t['accuracy']:.4f}")
    print(f"  Balanced Acc.   : {t['balanced_accuracy']:.4f}")
    print(f"  Precision       : {t['precision']:.4f}")
    print(f"  Recall          : {t['recall']:.4f}")
    print(f"  F1              : {t['f1']:.4f}")
    print(f"  AUC-ROC         : {t['auc']:.4f}   PR-AUC {t['pr_auc']:.4f}")
    print(f"  Specificity     : {t['specificity']:.4f}   ECE {t['ece']:.4f}")
    print(f"  confusion       : TN={t['tn']} FP={t['fp']} FN={t['fn']} TP={t['tp']}")


## 6. Single-case explanations

In [ ]:
# Single-case explanation figures (one case per figure, as per the paper).
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("/kaggle/working/outputs/*/*.png")):
    print(p)
    display(Image(filename=p))
